# EDA Universal
### Angka cepat + grafik opsional · tabular & NLP
**Electric blue `#0085ff` / sunflower `#fcd12a`**

Pilih visual berdasarkan pertanyaan data, bukan wajib menjalankan semua chart.

| Bagian | Pertanyaan | Visual |
|---|---|---|
| Fondasi · 00–03 | Data dan kolomnya apa? | Konfigurasi, pembaca, fitur bersama |
| Kualitas · 04–05, 17–18, 31–32 | Missing, noise, duplikat, leakage? | Lollipop, matriks, profil noise, overlap |
| Distribusi · 06–11, 24–28 | Seimbang? Panjangnya berbeda? | Donut, Pareto, violin + titik, ridgeline, hexbin |
| Waktu & sumber · 12–13, 29–30 | Lonjakan, pergeseran, dominasi grup? | Area, heatmap, stacked area, kurva konsentrasi |
| Bahasa · 15–16, 33–35 | Kata khas, coverage, struktur kosakata? | N-gram, Zipf, butterfly, kurva coverage, bubble |
| Struktur teks · 20–23, 36–39 | Pasangan, multilabel, aspek, tag, topik? | Scatter, ko-occurence, jaringan, proyeksi leksikal, transition map |
| Numerik · 14, 40 | Relasi fitur dan outlier? | Korelasi, bubble, boxen + strip |
| Shift · 19, 41 | Train dan test berbeda di mana? | ECDF dan dumbbell |

**Cara pakai**
1. Isi konfigurasi, jalankan 00–03, lalu pilih template. Dependensi tambahan disebut tepat di atas chart.
2. Untuk final, salin 00–03 dan chart pilihan. Helper visual pendek dipakai bersama agar kode chart tidak berulang.
3. `TRAIN=None` menghasilkan **preview sintetis** dengan variasi panjang, topik, sumber, dan tanggal. Bukan temuan data asli.

Statistik dasar tidak dibungkus framework; pakai `df.describe()` atau `df.info()` bila perlu. Tidak ada ekspor HTML, training model, atau file laporan otomatis.

**Mode kejar waktu:** RUN_PLOTS=False secara default. Run All menghasilkan **angka/tabel saja**. Setiap bagian grafik punya sel angka di atasnya. Grafik tersimpan pada sel opsional terlipat; ubah RUN_PLOTS=True untuk menampilkannya.

**Cakupan angka:** skema/missing/pseudo-null/unik, duplikat/konflik label, panjang/kuantil/coverage, top words, hapax/dis-legomena, n-gram, kata aneh, bahasa, setiap noise, sumber/label/waktu, train-test, dan task khusus.

Hitungan dasar/noise/duplikat memakai seluruh baris. Kosakata/n-gram maksimal SAMPLE teks terisi; bahasa maksimal LANG_SAMPLE. Penyebut persen dan batas tabel ditulis per bagian. TOP membatasi tampilan, bukan penyebut. NaN statistik berarti tidak tersedia/penyebut 0, bukan 0%. Output awal masih preview sintetis.


## 00 · Konfigurasi

Bagian utama yang diganti saat pindah dataset. `TARGET` bisa nama kolom atau list kolom biner untuk multilabel. Path relatif dihitung dari folder notebook. Gunakan path absolut agar bebas lokasi kernel.

In [1]:
TRAIN, TEST = None, None  # contoh: r"D:\data\train.csv", r"D:\data\test.csv"
TEXT, TEXT_B = "text", None
TARGET, TIME, SOURCE = "label", "time", "source"
TAGS = None  # kolom list tag BIO atau string tag dipisah spasi
LABEL_DROP = ["empty"]  # nilai yang dianggap belum berlabel
SAMPLE, TOP, SEED = 10_000, 15, 42
FREQ = "W"  # D: harian, W: mingguan, MS: bulanan
STOPWORDS = set("yang dan di ke dari untuk ini itu dengan pada adalah a the of to and".split())
ASPECTS = {}  # opsional, regex: {"harga": r"mahal|murah", "layanan": r"ramah|lambat"}
GROUP_COL, VALUE = "group", "rating"  # metadata opsional untuk konsentrasi dan numerik
PROJECTION_N = 1500  # batas proyeksi leksikal; tidak perlu embedding pretrained
RUN_PLOTS = False  # False: Run All hanya angka/tabel; True: grafik juga
KIND = "category"  # "numeric" untuk regresi
LANG_SAMPLE, LANG_MIN_WORDS, LANG_MIN_SCORE = 500, 5, .8
ID_COL = None  # opsional: kolom ID baris
KNOWN_WORDS = set()  # opsional: kosakata referensi; kosong = tidak menilai OOV/typo
TOKENIZER_PATH = None  # opsional: folder tokenizer lokal, tanpa download
MAX_LENGTHS = [64,128,256,512]


## 01 · Setup

Dependensi: `pandas`, `numpy`, `matplotlib`, `seaborn`. N-gram memakai `scikit-learn`; Excel/parquet membutuhkan engine pembacanya. Helper hanya mengatur tampilan, bar chart, dan heatmap.

In [2]:
from pathlib import Path
from collections import Counter
import re
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import PercentFormatter

BLUE, YELLOW, INK = "#0085ff", "#fcd12a", "#14243b"
# Transparansi hanya dipakai untuk overlap; warna utama tetap jenuh.
CMAP = LinearSegmentedColormap.from_list("eda", ["#ffffff", BLUE])
DIVERGE = LinearSegmentedColormap.from_list("signed", [YELLOW, "#ffffff", BLUE])
sns.set_theme(style="whitegrid", palette=[BLUE, YELLOW], font_scale=.95)
plt.rcParams.update({"figure.figsize": (10, 4.8), "figure.dpi": 120,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.titleweight": "bold", "axes.titlepad": 18, "axes.titlelocation": "left",
    "text.color": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "grid.alpha": .18, "grid.linewidth": .7, "font.family": "DejaVu Sans"})

In [3]:
def finish(title, xlabel=None, ylabel=None):
    plt.title(title)
    if xlabel is not None: plt.xlabel(xlabel)
    if ylabel is not None: plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()
    plt.close()

def bars(values, title, xlabel="Jumlah", color=BLUE):
    values = values.dropna().nlargest(TOP).sort_values()
    if values.empty:
        print(f"Lewati: {title} — tidak ada nilai.")
        return
    fig, ax = plt.subplots(figsize=(10, max(3.3, len(values) * .38)))
    ax.hlines(range(len(values)), 0, values, color=color, linewidth=3, alpha=.55)
    ax.scatter(values, range(len(values)), s=85, color=color, zorder=3)
    ax.set_yticks(range(len(values)), values.index.astype(str))
    for i, value in enumerate(values):
        ax.annotate(f"{value:,.1f}".removesuffix(".0"), (value, i), xytext=(9, 0),
                    textcoords="offset points", va="center", fontsize=9, weight="bold")
    ax.set_xlim(0, max(1, values.max()) * 1.23)
    ax.grid(axis="y", visible=False)
    finish(title, xlabel, "")

def heat(table, title, percent=False):
    if table.empty:
        print(f"Lewati: {title} — tidak ada nilai.")
        return
    plt.figure(figsize=(max(7, table.shape[1] * .62), max(3.5, table.shape[0] * .43)))
    sns.heatmap(table, cmap=CMAP, annot=table.size <= 150, linewidths=2,
                fmt=".0f" if percent else ".2f", cbar_kws={"shrink": .65})
    finish(title)

## 02 · Muat data

Mendukung CSV, TSV, JSON, JSONL, Excel, parquet. Pembaca tidak membuang baris rusak diam-diam; sesuaikan `read_data` bila format data berbeda.

In [4]:
def read_data(path):
    suffix = Path(path).suffix.lower()
    if suffix == ".parquet": return pd.read_parquet(path)
    if suffix in {".xlsx", ".xls"}: return pd.read_excel(path)
    if suffix in {".json", ".jsonl"}: return pd.read_json(path, lines=suffix == ".jsonl")
    return pd.read_csv(path, sep="\t" if suffix == ".tsv" else ",", low_memory=False)

if TRAIN is None:

    rng = np.random.default_rng(SEED)
    topics = ["layanan cepat ramah admin respons bantuan", "harga mahal diskon murah promo biaya",
              "produk kualitas bagus rusak bahan ukuran", "kirim kurir paket lambat tiba pengiriman"]
    rows = []
    for i in range(1200):
        topic = rng.integers(4)
        label = rng.choice(["positif", "netral", "negatif"], p=[.52, .28, .20])
        words = rng.choice(topics[topic].split() + ["saya", "dan", "ini", "untuk", "tidak", "sudah"],
                           size=min(140, max(2, int(rng.lognormal(2.8, .7)))))
        value = " ".join(words)
        if i % 9 == 0: value += " @admin!!! https://contoh.id #review"
        if i % 31 == 0: value = None
        if i % 37 == 0: value = "produk bagus kualitas baik"
        rows.append((value, label, rng.choice(["web", "aplikasi", "sosmed"], p=[.18, .52, .30]),
                     pd.Timestamp("2025-01-01") + pd.Timedelta(hours=int(rng.beta(2, 4) * 5000)),
                     rng.integers(1, 6), f"thread-{int(rng.zipf(2)) % 80}", topics[topic]))
    df = pd.DataFrame(rows, columns=["text", "label", "source", "time", "rating", "group", "text_b"])
    df.loc[df.index % 17 == 0, "label"] = None
    df.loc[df.index % 13 == 0, "rating"] = np.nan
    print("PREVIEW: data sintetis; bukan temuan dataset asli.")
else:
    df = read_data(TRAIN).reset_index(drop=True)
test = read_data(TEST).reset_index(drop=True) if TEST else None
if df.empty: raise ValueError("Dataset kosong: isi data sebelum membuat chart.")
print(f"{len(df):,} baris × {df.shape[1]} kolom")

PREVIEW: data sintetis; bukan temuan dataset asli.
1,200 baris × 7 kolom


## 03 · Fitur bersama

`text`, `features`, `sample_text`, `labels`, `dates` disiapkan sekali. Isi kolom opsional dengan `None` untuk menonaktifkan. Teks null dianggap kosong hanya untuk perhitungan panjang; data asli tetap ada di `df`.

In [5]:
has = lambda col: isinstance(col, str) and col in df.columns
text = df[TEXT].astype("string").fillna("") if has(TEXT) else pd.Series("", index=df.index)
features = pd.DataFrame({"karakter": text.str.len(), "kata": text.str.split().str.len()})
sample_text = text[text.str.strip().ne("")].sample(min(SAMPLE, text.str.strip().ne("").sum()), random_state=SEED)
targets = [TARGET] if isinstance(TARGET, str) else list(TARGET or [])
targets = [col for col in targets if has(col)]
labels = df[targets[0]].replace(LABEL_DROP, np.nan) if len(targets) == 1 else None
# Campuran format tanggal didukung pandas >= 2; tanggal gagal parse menjadi NaT.
dates = pd.to_datetime(df[TIME], errors="coerce", format="mixed", utc=True) if has(TIME) else None
print("Teks:", TEXT if has(TEXT) else "tidak ada", "| Target:", targets or "tidak ada")
if dates is not None: print(f"Tanggal tidak valid/kosong: {dates.isna().sum():,}")
lexical = features.copy()
lexical["unik_%"] = text.str.lower().str.split().map(lambda w: 100 * len(set(w)) / max(1, len(w)))
lexical["digit_%"] = text.str.count(r"\d").div(features["karakter"].clip(lower=1)).mul(100)
lexical["kapital_%"] = text.str.count(r"[A-Z]").div(features["karakter"].clip(lower=1)).mul(100)


Teks: text | Target: ['label']
Tanggal tidak valid/kosong: 0


### 03a · Alat angka singkat
Semua sel angka memakai 00–03c, lalu bisa dipilih sendiri. Helper hanya menampilkan tabel dan menghitung persentase. `NaN` pada statistik berarti tidak tersedia/penyebut 0, bukan 0%.

In [6]:
def show(title, data):
    print(f"\n{title}")
    with pd.option_context("display.max_rows",None,"display.max_columns",None,"display.max_colwidth",140):
        display(data.to_frame().round(3) if isinstance(data,pd.Series) else data.round(3))

def pct(n, total):
    return 100*n/total if total else np.nan

def distribution(values):
    count = values.value_counts(dropna=False)
    return pd.DataFrame({"n":count,"%":count.div(len(values) or np.nan).mul(100)})

def flag_rates(flags):
    return pd.DataFrame({"n":flags.sum(),"%":flags.mean()*100}).sort_values("n",ascending=False)

def unique_count(values):
    try: return values.nunique(dropna=True)
    except TypeError: return values.dropna().map(repr).nunique()

### 03b · Kosakata dihitung sekali
Maksimal `SAMPLE` baris teks terisi, acak dengan SEED. Duplikat tetap berbobot satu per baris. Untuk seluruh teks, isi SAMPLE setidaknya sebesar jumlah baris.

Token kosakata = rangkaian huruf Unicode setelah casefold; angka, emoji, underscore dan tanda baca menjadi pemisah. Panjang **kata** di features memakai split spasi. Keduanya bukan token model. Hitungan kosakata sebelum stopword removal. N-gram mengikuti urutan asli dalam satu dokumen, tidak menyambung kata melewati stopword.

In [7]:
word_lists = sample_text.str.casefold().str.findall(r"[^\W\d_]+")
vocab_counts = pd.Series(Counter(w for seq in word_lists for w in seq),dtype="int64").sort_values(ascending=False)
doc_counts = pd.Series(Counter(w for seq in word_lists for w in set(seq)),dtype="int64")
word_table = vocab_counts.rename("kemunculan").to_frame()
word_table["%_token"] = vocab_counts.div(vocab_counts.sum() or np.nan).mul(100)
word_table["dokumen"] = doc_counts.reindex(word_table.index)
word_table["%_dokumen"] = word_table.dokumen.div(len(word_lists) or np.nan).mul(100)
normalized_text = text.str.casefold().str.replace(r"\s+"," ",regex=True).str.strip()
filled_text = text.str.strip().ne("")
print(f"Kosakata: {len(word_lists):,}/{filled_text.sum():,} teks terisi; {vocab_counts.sum():,} token; {len(vocab_counts):,} tipe kata.")

Kosakata: 1,163/1,163 teks terisi; 25,352 token; 35 tipe kata.


### 03c · Flag noise dihitung sekali
Seluruh baris, tanpa mengubah data. Satu teks bisa memiliki banyak flag; persentase tidak harus berjumlah 100%. Regex memberi indikator: non-ASCII bisa valid, rentang emoji juga mencakup sebagian simbol, indikasi encoding rusak perlu diperiksa contohnya.

In [8]:
NOISE = {
    "URL":r"https?://|www\.", "email":r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}",
    "mention":r"(?<!\w)@\w+", "hashtag":r"#\w+", "angka":r"\d",
    "huruf+angka":r"\b(?=\w*[a-zA-Z])(?=\w*\d)\w+\b",
    "emoji/simbol":r"[\U0001F1E6-\U0001FAFF\u2600-\u27BF]", "non-ASCII":r"[^\x00-\x7f]",
    "huruf berulang >=3":r"([^\W\d_])\1{2,}", "baca berulang":r"[!?.,]{2,}", "KAPITAL >=3":r"\b[A-Z]{3,}\b",
    "HTML tag":r"</?[a-zA-Z][^>]*>", "HTML entity":r"&(?:#\d+|#x[\da-fA-F]+|[a-zA-Z]+);",
    "indikasi encoding rusak":r"\ufffd|Ã.|Â.|â€", "invisible/bidi":r"[\u200b\u200e\u200f\u202a-\u202e\u2066-\u2069\ufeff]",
    "kontrol":r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "newline/tab":r"[\n\r\t]",
    "spasi berulang":r" {2,}", "spasi tepi":r"^\s|\s$",
    "negasi":r"\b(?:tidak|tak|bukan|belum|jangan|tanpa|nggak|ngga|gak|ga|gk|not|never|no)\b",
    "slang contoh":r"\b(?:gk|gak|nggak|ngga|bgt|yg|dgn|utk|udh|klo|jg|tdk|blm)\b",
    "kata ulang berurutan":r"\b(\w+)\s+\1\b"}
noise_flags = pd.DataFrame({k:text.str.count(v,flags=0 if k=="KAPITAL >=3" else re.I).gt(0)
                           for k,v in NOISE.items()},index=df.index)

## 04 · Missing values

Pakai `df`. Persentase dihitung pada seluruh baris; maksimal `TOP` kolom paling banyak kosong.

**Angka cepat · skema, missing, kosong, unik**

Seluruh data. NaN, kosong/spasi, pseudo-null dibedakan. Nilai unik tidak menghitung NaN. Semua kolom ditampilkan.

In [9]:
blank = df.apply(lambda s:s.astype("string").str.fullmatch(r"\s*",na=False).sum())
pseudo = df.apply(lambda s:s.astype("string").str.strip().str.casefold().isin(["nan","null","none","n/a","na","-","[sticker]"]).sum())
schema = pd.DataFrame({"tipe":df.dtypes.astype(str),"terisi":df.notna().sum(),"NaN":df.isna().sum(),
    "NaN_%":df.isna().mean()*100,"kosong/spasi":blank,"pseudo_null":pseudo,"unik_non_null":df.apply(unique_count)})
schema["unik_%_terisi"] = schema.unik_non_null.div(schema.terisi.replace(0,np.nan))*100
show("Skema seluruh kolom",schema)
print(f"Baris={len(df):,}; kolom={df.shape[1]}; memori={df.memory_usage(deep=True).sum()/1e6:.2f} MB")
print("Kolom konstan/semua-null:",schema.index[schema.unik_non_null.le(1)].tolist())
if has(ID_COL): show("ID",pd.Series({"null":df[ID_COL].isna().sum(),"unik":unique_count(df[ID_COL]),"duplikat_ekstra":df[ID_COL].dropna().duplicated().sum()}))


Skema seluruh kolom

,tipe,terisi,NaN,NaN_%,kosong/spasi,pseudo_null,unik_non_null,unik_%_terisi
text,object,1163,37,3.083,0,0,1131,97.248
label,object,1129,71,5.917,0,0,3,0.266
source,object,1200,0,0.000,0,0,3,0.250
time,datetime64[ns],1200,0,0.000,0,0,1009,84.083
rating,float64,1107,93,7.750,0,0,5,0.452
group,object,1200,0,0.000,0,0,38,3.167
text_b,object,1200,0,0.000,0,0,4,0.333


Baris=1,200; kolom=7; memori=0.59 MB
Kolom konstan/semua-null: []


**Angka cepat · integritas teks**

Seluruh baris. Non-string ditandai sebelum casting untuk analisis. Data asli tetap utuh.

In [10]:
if has(TEXT):
    quality = pd.DataFrame({"NaN":df[TEXT].isna(),"non_string_non_null":df[TEXT].map(lambda v:not isinstance(v,str)) & df[TEXT].notna(),
        "kosong/spasi_non_null":~filled_text & df[TEXT].notna(),
        "pseudo_null":text.str.strip().str.casefold().isin(["nan","null","none","n/a","na","-","[sticker]"]),
        "terisi_tanpa_huruf":filled_text & ~text.str.contains(r"[^\W\d_]"),"hanya_angka":text.str.fullmatch(r"\s*\d+\s*",na=False)})
    show("Kondisi teks · % seluruh baris",flag_rates(quality))


Kondisi teks · % seluruh baris


,n,%
NaN,37,3.083
non_string_non_null,0,0.0
kosong/spasi_non_null,0,0.0
pseudo_null,0,0.0
terisi_tanpa_huruf,0,0.0
hanya_angka,0,0.0


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [11]:
if RUN_PLOTS:
    bars(df.isna().mean().mul(100), "Nilai kosong per kolom", "% baris", YELLOW)

## 05 · Pola missing

Pakai `df`. Heatmap maksimal 40 kolom dan 100 baris sampel; pola visual ini tidak mewakili urutan waktu.

**Angka cepat · kelengkapan baris**

Seluruh baris. Pola NaN teratas pada maksimal 40 kolom, sesuai batas lebar grafik.

In [12]:
show("Jumlah kolom missing per baris",distribution(df.isna().sum(axis=1)).sort_index())
print(f"Baris lengkap={df.notna().all(axis=1).sum():,} ({df.notna().all(axis=1).mean():.2%})")
show("Pola NaN teratas · True=missing",df.iloc[:,:40].isna().value_counts().head(TOP).rename("n").to_frame())


Jumlah kolom missing per baris


,n,%
0,1009,84.083
1,181,15.083
2,10,0.833


Baris lengkap=1,009 (84.08%)

Pola NaN teratas · True=missing


n
text  label source time  rating group text_b      
False False False  False False  False False   1009
                         True   False False     85
      True  False  False False  False False     63
True  False False  False False  False False     33
False True  False  False True   False False      6
True  False False  False True   False False      2
      True  False  False False  False False      2

**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [13]:
if RUN_PLOTS:
    view = df.sample(min(100, len(df)), random_state=SEED).iloc[:, :40]
    plt.figure(figsize=(12, 4))
    sns.heatmap(view.isna(), cmap=[BLUE, YELLOW], cbar=False, yticklabels=False)
    finish("Pola missing · biru = terisi, kuning = kosong", "Kolom", "Baris sampel")

## 06 · Frekuensi kategori / sumber

Pakai `df`. Ganti `COL` untuk kategori lain. Nilai kosong ditampilkan tersendiri.

**Angka cepat · distribusi kategori**

Ganti COL. NaN ikut dihitung; % memakai seluruh baris. TOP membatasi tampilan.

In [14]:
COL = SOURCE
if has(COL):
    print("Kategori unik non-null:",unique_count(df[COL]))
    show(f"Distribusi {COL} · % seluruh baris",distribution(df[COL]).head(TOP))
else: print("Lewati: kolom kategori tidak ada.")

Kategori unik non-null: 3

Distribusi source · % seluruh baris


,n,%
source,,
aplikasi,621,51.750
sosmed,358,29.833
web,221,18.417


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [15]:
if RUN_PLOTS:
    COL = SOURCE
    if has(COL):
        bars(df[COL].astype("string").fillna("(kosong)").value_counts(), f"Frekuensi · {COL}")
    else:
        print("Lewati: kolom kategori belum tersedia.")

## 07 · Distribusi target

Bar plot jumlah baris per label, maksimal `TOP` label terpopuler. Nilai kosong dilewati. Gunakan `KIND = "numeric"` untuk histogram target regresi.

**Angka cepat · target & imbalance**

KIND di konfigurasi berlaku untuk angka/grafik. Kategori menampilkan % baris berlabel dan % seluruh data. Regresi melaporkan gagal konversi.

In [16]:
if labels is None: print("Lewati: perlu satu target.")
elif KIND=="numeric":
    values = pd.to_numeric(labels,errors="coerce").replace([np.inf,-np.inf],np.nan)
    print("Gagal numerik/non-finite:",(labels.notna() & values.isna()).sum())
    show("Target numerik",values.describe(percentiles=[.01,.05,.5,.95,.99]))
else:
    count = labels.dropna().value_counts()
    print(f"Berlabel={labels.notna().sum():,} ({labels.notna().mean():.2%}); kelas={len(count):,}")
    show("Distribusi label",count.rename("n").to_frame().assign(pct_berlabel=count/(count.sum() or np.nan)*100,pct_semua=count/len(df)*100).head(TOP))
    print("Rasio kelas terbesar/terkecil:",count.max()/count.min() if len(count) else np.nan)

Berlabel=1,129 (94.08%); kelas=3

Distribusi label


,n,pct_berlabel,pct_semua
label,,,
positif,576,51.019,48.000
netral,336,29.761,28.000
negatif,217,19.221,18.083


Rasio kelas terbesar/terkecil: 2.654377880184332


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [17]:
if RUN_PLOTS:
    if labels is None or labels.dropna().empty:
        print("Lewati: perlu satu target terisi.")
    elif KIND == "numeric":
        values = pd.to_numeric(labels, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(values):
            plt.figure()
            sns.histplot(values, bins=30, color=BLUE)
            finish("Distribusi target numerik", targets[0], "Jumlah")
    else:
        counts = labels.dropna().astype(str).value_counts().head(TOP)
        counts.sort_values().plot.barh(color=BLUE)
        finish("Distribusi target", "Jumlah baris berlabel", "Label")

## 08 · Komposisi label per sumber

Pakai `labels`, `SOURCE`. Tiap baris heatmap berjumlah 100%; maksimal `TOP` sumber dan label terpopuler, persentase dihitung sebelum pembatasan.

**Angka cepat · label per sumber**

Klasifikasi satu label; hanya label/sumber terisi. Persen per sumber memakai semua label sebelum membatasi TOP.

In [18]:
if KIND=="category" and labels is not None and has(SOURCE):
    table = pd.crosstab(df[SOURCE],labels)
    rows,cols = table.sum(axis=1).nlargest(TOP).index,table.sum().nlargest(TOP).index
    show("Jumlah label per sumber",table.loc[rows,cols])
    show("Komposisi · % baris berlabel per sumber",table.div(table.sum(axis=1),axis=0).mul(100).loc[rows,cols])


Jumlah label per sumber


label,positif,netral,negatif
source,,,
aplikasi,297,177,115
sosmed,162,101,69
web,117,58,33



Komposisi · % baris berlabel per sumber


label,positif,netral,negatif
source,,,
aplikasi,50.424,30.051,19.525
sosmed,48.795,30.422,20.783
web,56.250,27.885,15.865


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [19]:
if RUN_PLOTS:
    if labels is not None and has(SOURCE):
        table = pd.crosstab(df[SOURCE], labels, normalize="index").mul(100)
        rows = df[SOURCE].value_counts().head(TOP).index
        cols = labels.value_counts().head(TOP).index
        heat(table.reindex(index=rows, columns=cols).fillna(0), "Komposisi label per sumber (%)", True)
    else:
        print("Lewati: perlu satu target dan sumber.")

## 09 · Histogram panjang teks

Pakai `features`, `TEXT`. Garis kuning menunjukkan P95 jumlah kata, bukan token model. Teks kosong tetap dihitung.

**Angka cepat · panjang teks**

Seluruh baris termasuk kosong/null dengan panjang 0. Kata di sini split spasi, bukan token model.

In [20]:
if has(TEXT):
    show("Panjang karakter/kata",features.describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99,.999]).T)
    print(f"Total karakter={features.karakter.sum():,}; kata/spasi={features.kata.sum():,}")
    extremes = pd.concat([features.kata.nsmallest(5),features.kata.nlargest(5)]).index.unique()
    show("Terpendek & terpanjang",features.loc[extremes].assign(teks=text.loc[extremes].str.slice(0,140)))


Panjang karakter/kata


,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,99.9%,max
karakter,1200.0,125.346667,109.017906,0.0,0.0,21.0,54.0,97.5,162.25,248.2,336.3,559.32,836.07,867.0
kata,1200.0,20.916667,18.192779,0.0,0.0,4.0,9.0,16.0,27.0,42.0,56.0,92.05,140.0,140.0


Total karakter=150,416; kata/spasi=25,100

Terpendek & terpanjang


,karakter,kata,teks
31,0,0,
62,0,0,
93,0,0,
124,0,0,
155,0,0,
273,780,140,saya saya harga murah dan ini ini sudah ini saya murah promo untuk biaya ini mahal mahal diskon diskon dan dan saya murah saya mahal bia...
470,867,140,produk ukuran sudah untuk tidak kualitas ukuran sudah untuk ukuran produk rusak saya dan sudah kualitas bahan rusak sudah kualitas bagus...
953,850,140,bantuan admin ramah cepat saya sudah saya sudah untuk ini ini sudah tidak cepat dan bantuan tidak ramah untuk bantuan sudah untuk respon...
13,758,125,tidak layanan bantuan dan respons cepat ramah untuk admin admin sudah cepat tidak sudah dan cepat cepat admin saya cepat bantuan cepat s...
228,705,121,ini dan kualitas sudah dan untuk dan bahan bahan ini tidak kualitas tidak untuk kualitas kualitas ini tidak bahan dan sudah rusak bagus ...


**Angka cepat · tokenizer kandidat · opsional**

Isi TOKENIZER_PATH dengan folder lokal; membutuhkan transformers. Sampel kosakata, termasuk special tokens, tanpa truncation/download.

In [21]:
if TOKENIZER_PATH and len(sample_text):
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH,local_files_only=True)
    token_lengths = sample_text.map(lambda s:len(tokenizer.encode(s,add_special_tokens=True,truncation=False)))
    show("Panjang token model",token_lengths.describe(percentiles=[.5,.9,.95,.99]))
    show("Melampaui batas token · % sampel",pd.DataFrame({"n_terpotong":{k:token_lengths.gt(k).sum() for k in MAX_LENGTHS},
         "%_terpotong":{k:token_lengths.gt(k).mean()*100 for k in MAX_LENGTHS}}))
else: print("Token model dilewati: isi TOKENIZER_PATH bila perlu.")

Token model dilewati: isi TOKENIZER_PATH bila perlu.


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [22]:
if RUN_PLOTS:
    if has(TEXT):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for ax, col in zip(axes, features):
            sns.histplot(features[col], bins=35, color=BLUE, ax=ax)
            ax.axvline(features[col].quantile(.95), color=YELLOW, lw=3, label="P95")
            ax.set(title=f"Panjang · {col}", ylabel="Jumlah")
            ax.legend()
        plt.tight_layout(); plt.show(); plt.close()
    else:
        print("Lewati: kolom teks belum tersedia.")

## 10 · ECDF panjang teks

Pakai `features`. Baca proporsi teks yang muat pada suatu batas kata; tidak memakai estimasi token.

**Angka cepat · coverage panjang**

Seluruh data. Batas memakai kata/spasi; % teks terisi terpisah agar baris kosong tidak menaikkan cakupan.

In [23]:
if has(TEXT):
    limits = [0,1,5,10,20,50,100,200,500]
    show("Teks <= batas kata",pd.DataFrame({"batas":limits,"n_semua":[features.kata.le(k).sum() for k in limits],
        "%_semua":[features.kata.le(k).mean()*100 for k in limits],"%_terisi":[features.loc[filled_text,"kata"].le(k).mean()*100 for k in limits]}))


Teks <= batas kata


,batas,n_semua,%_semua,%_terisi
0,0,37,3.083,0.000
1,1,37,3.083,0.000
2,5,148,12.333,9.544
3,10,366,30.500,28.289
4,20,752,62.667,61.479
5,50,1120,93.333,93.121
6,100,1190,99.167,99.140
7,200,1200,100.000,100.000
8,500,1200,100.000,100.000


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [24]:
if RUN_PLOTS:
    if has(TEXT):
        plt.figure()
        sns.ecdfplot(x=features["kata"], color=BLUE, linewidth=2)
        plt.axhline(.95, color=YELLOW, lw=2, label="95% teks")
        plt.legend()
        finish("Cakupan panjang teks", "Jumlah kata", "Proporsi kumulatif")

## 11 · Violin + titik sampel

Pakai `features`, `df`. Bentuk violin menunjukkan distribusi seluruh data, garis dalam menunjukkan kuartil; titik kuning maksimal 350 sampel. Kelompok dengan data sedikit jangan diinterpretasikan berlebihan.

**Angka cepat · panjang per kelompok**

Seluruh baris, NaN kelompok ikut ditampilkan. Ganti COL ke sumber/target kategori.

In [25]:
COL = SOURCE
if has(TEXT) and has(COL):
    show(f"Panjang kata per {COL}",features.kata.groupby(df[COL],dropna=False).agg(n="size",mean="mean",median="median",
        p90=lambda s:s.quantile(.9),p95=lambda s:s.quantile(.95),maks="max").sort_values("n",ascending=False).head(TOP))


Panjang kata per source


,n,mean,median,p90,p95,maks
source,,,,,,
aplikasi,621,20.374,16.0,39.0,52.0,140
sosmed,358,21.765,16.0,45.0,59.0,121
web,221,21.068,16.0,40.0,57.0,140


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [26]:
if RUN_PLOTS:
    GROUP = SOURCE  # ganti ke kolom target untuk perbandingan label
    if has(TEXT) and has(GROUP):
        view = features.assign(kelompok=df[GROUP].astype("string")).dropna()
        order = view["kelompok"].value_counts().head(min(TOP, 8)).index.tolist()
        if order:
            fig, ax = plt.subplots(figsize=(10, max(4, len(order) * .7)))
            sns.violinplot(data=view, x="kata", y="kelompok", order=order, inner="quart",
                           color=BLUE, cut=0, linewidth=1, ax=ax)
            dots = view.sample(min(350, len(view)), random_state=SEED)
            sns.stripplot(data=dots, x="kata", y="kelompok", order=order, color=YELLOW,
                          size=3, alpha=.75, jitter=.16, ax=ax)
            finish(f"Distribusi panjang per {GROUP}", "Jumlah kata · titik = sampel", "")
    else:
        print("Lewati: perlu teks dan kolom kelompok.")

## 12 · Volume waktu

Pakai `dates`. Agregasi berdasarkan `FREQ` dan zona UTC; tanggal tidak valid dikeluarkan.

**Angka cepat · rentang & volume waktu**

UTC. Persen periode memakai tanggal valid. Tanggal gagal parse dikeluarkan.

In [27]:
if dates is not None:
    print(f"Valid={dates.notna().sum():,}; kosong/gagal={dates.isna().sum():,}; awal={dates.min()}; akhir={dates.max()}")
    if dates.notna().any():
        volume = pd.Series(1,index=pd.DatetimeIndex(dates.dropna())).resample(FREQ).sum()
        show("Volume per periode",volume.describe())
        show("Periode teramai",volume.nlargest(TOP).rename("n").to_frame().assign(pct=volume.nlargest(TOP)/volume.sum()*100))
        print("Periode tanpa baris:",volume.eq(0).sum())

Valid=1,200; kosong/gagal=0; awal=2025-01-01 20:00:00+00:00; akhir=2025-06-30 18:00:00+00:00

Volume per periode


,0
count,27.000
mean,44.444
std,30.788
min,2.000
25%,21.500
50%,38.000
75%,72.000
max,95.000



Periode teramai

,n,pct
time,,
2025-03-09 00:00:00+00:00,95,7.917
2025-02-09 00:00:00+00:00,91,7.583
2025-03-02 00:00:00+00:00,87,7.250
2025-02-23 00:00:00+00:00,82,6.833
2025-02-02 00:00:00+00:00,80,6.667
2025-03-16 00:00:00+00:00,78,6.500
2025-03-23 00:00:00+00:00,77,6.417
2025-02-16 00:00:00+00:00,67,5.583
2025-01-26 00:00:00+00:00,64,5.333


Periode tanpa baris: 0


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [28]:
if RUN_PLOTS:
    if dates is not None and dates.notna().any():
        volume = pd.Series(1, index=pd.DatetimeIndex(dates.dropna())).resample(FREQ).sum()
        plt.figure(figsize=(11, 4))
        plt.plot(volume.index, volume.values, color=BLUE, lw=2)
        plt.fill_between(volume.index, volume.values, color=BLUE, alpha=.15)
        finish(f"Volume data · {FREQ} (UTC)", "Waktu", "Jumlah baris")
    else:
        print("Lewati: tidak ada tanggal valid.")

## 13 · Heatmap hari × jam

Pakai `dates`. Zona UTC; ubah `.dt.tz_convert("Asia/Jakarta")` jika ingin waktu lokal.

**Angka cepat · distribusi hari & jam**

UTC. Persen memakai jumlah tanggal valid; NaT dikeluarkan.

In [29]:
if dates is not None:
    show("Hari · 0=Senin, 6=Minggu",distribution(dates.dropna().dt.dayofweek).sort_index())
    show("Jam UTC",distribution(dates.dropna().dt.hour).sort_index())
    show("Jumlah hari × jam",pd.crosstab(dates.dt.dayofweek,dates.dt.hour).reindex(index=range(7),columns=range(24),fill_value=0))


Hari · 0=Senin, 6=Minggu


,n,%
time,,
0,173,14.417
1,175,14.583
2,194,16.167
3,175,14.583
4,168,14.000
5,164,13.667
6,151,12.583



Jam UTC


,n,%
time,,
0,43,3.583
1,49,4.083
2,43,3.583
3,49,4.083
4,46,3.833
5,60,5.000
6,53,4.417
7,42,3.500
8,57,4.750



Jumlah hari × jam

time,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
time,,,,,,,,,,,,,,,,,,,,,,,,
0,6,10,4,8,6,9,9,5,7,10,7,7,3,11,8,5,9,5,7,10,6,7,11,3
1,5,9,8,12,7,6,8,7,6,4,5,8,8,12,5,9,4,7,8,8,5,3,11,10
2,5,7,8,8,7,13,9,5,15,9,11,12,10,6,5,6,8,5,6,5,9,9,6,10
3,7,6,4,6,4,10,9,3,10,8,8,7,8,7,8,11,10,8,5,4,8,7,6,11
4,7,5,10,2,11,7,4,11,5,5,5,5,5,10,6,11,8,6,9,7,10,11,5,3
5,7,5,2,7,6,4,10,7,7,7,9,8,9,6,5,7,8,10,4,4,10,3,9,10
6,6,7,7,6,5,11,4,4,7,2,4,7,7,6,8,8,4,6,7,6,3,10,6,10


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [30]:
if RUN_PLOTS:
    if dates is not None and dates.notna().any():
        local = dates  # dates.dt.tz_convert("Asia/Jakarta")
        table = pd.crosstab(local.dt.dayofweek, local.dt.hour).reindex(index=range(7), columns=range(24), fill_value=0)
        table.index = ["Sen", "Sel", "Rab", "Kam", "Jum", "Sab", "Min"]
        heat(table, "Volume menurut hari dan jam (UTC)", True)

## 14 · Korelasi numerik

Pakai `df`. Isi `COLS` untuk memilih fitur relevan dan menghindari ID numerik. Korelasi bukan bukti sebab-akibat. Kuning = korelasi negatif, putih = nol, biru = positif.

**Angka cepat · numerik & korelasi**

Maksimal 20 kolom numerik; infinite dikeluarkan. Periksa ID numerik sebelum menginterpretasikan korelasi.

In [31]:
num = df.select_dtypes(include="number").iloc[:,:20].replace([np.inf,-np.inf],np.nan)
if num.shape[1]:
    show("Deskripsi numerik",num.describe(percentiles=[.01,.5,.99]).T)
    corr = num.loc[:,num.nunique().gt(1)].corr(method="spearman")
    show("Korelasi Spearman",corr)
    pairs_corr = corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool)).stack()
    show("Korelasi terkuat",pairs_corr.sort_values(key=abs,ascending=False).head(TOP).rename("rho").to_frame())


Deskripsi numerik


,count,mean,std,min,1%,50%,99%,max
rating,1107.0,2.938,1.421,1.0,1.0,3.0,5.0,5.0



Korelasi Spearman


,rating
rating,1.0



Korelasi terkuat


,,rho


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [32]:
if RUN_PLOTS:
    COLS = None  # contoh: ["harga", "rating", "likes"]
    num = df[COLS] if COLS else df.select_dtypes(include="number").iloc[:, :20]
    num = num.replace([np.inf, -np.inf], np.nan)
    num = num.loc[:, num.nunique() > 1]
    if num.shape[1] >= 2:
        corr = num.corr(method="spearman")
        plt.figure(figsize=(8, 6))
        sns.heatmap(corr, cmap=DIVERGE, center=0, vmin=-1, vmax=1, annot=True, fmt=".2f", square=True)
        finish("Korelasi Spearman")
    else:
        print("Lewati: perlu minimal dua kolom numerik bervariasi.")

## 15 · Kata & n-gram teratas

Pakai `sample_text`, `STOPWORDS`. Frekuensi kemunculan pada sampel; stopword dapat diubah di konfigurasi. Tidak membutuhkan model.

**Angka cepat · top words & document frequency**

Sampel word_lists. % token = kemunculan/seluruh token; % dokumen = dokumen yang memuat kata/semua baris sampel. STOPWORDS hanya daftar contoh.

In [33]:
show("Top kata termasuk stopword",word_table.head(TOP))
show("Top kata selain STOPWORDS",word_table.loc[~word_table.index.isin(STOPWORDS)].head(TOP))
print("Token termasuk STOPWORDS (%):",pct(vocab_counts[vocab_counts.index.isin(STOPWORDS)].sum(),vocab_counts.sum()))


Top kata termasuk stopword


,kemunculan,%_token,dokumen,%_dokumen
sudah,2104,8.299,844,72.571
untuk,2100,8.283,816,70.163
dan,2069,8.161,827,71.109
saya,2050,8.086,817,70.249
ini,2043,8.059,821,70.593
tidak,1937,7.640,829,71.281
admin,706,2.785,324,27.859
bagus,625,2.465,261,22.442
produk,620,2.446,259,22.270
ukuran,605,2.386,221,19.003



Top kata selain STOPWORDS


,kemunculan,%_token,dokumen,%_dokumen
sudah,2104,8.299,844,72.571
saya,2050,8.086,817,70.249
tidak,1937,7.640,829,71.281
admin,706,2.785,324,27.859
bagus,625,2.465,261,22.442
produk,620,2.446,259,22.270
ukuran,605,2.386,221,19.003
kualitas,594,2.343,243,20.894
ramah,591,2.331,221,19.003
rusak,583,2.300,220,18.917


Token termasuk STOPWORDS (%): 24.502997791101294


**Angka cepat · unigram, bigram, trigram**

Sampel. N-gram tidak melintasi batas dokumen atau melompati stopword. Persen kemunculan memakai total n-gram; persen dokumen memakai semua baris sampel.

In [34]:
ngram_summary = []
for size in (1,2,3):
    sequences = word_lists.map(lambda seq:[" ".join(seq[i:i+size]) for i in range(len(seq)-size+1)])
    count = pd.Series(Counter(g for seq in sequences for g in seq),dtype="int64").sort_values(ascending=False)
    documents = pd.Series(Counter(g for seq in sequences for g in set(seq)),dtype="int64")
    table = count.rename("kemunculan").to_frame().assign(pct=count/(count.sum() or np.nan)*100,
        dokumen=documents.reindex(count.index),pct_dokumen=documents.reindex(count.index)/(len(word_lists) or np.nan)*100)
    show(f"Top {size}-gram",table.head(TOP))
    ngram_summary.append({"n":size,"total":count.sum(),"unik":len(count),"hapax":count.eq(1).sum()})
show("Ringkasan n-gram",pd.DataFrame(ngram_summary).set_index("n"))


Top 1-gram


,kemunculan,pct,dokumen,pct_dokumen
sudah,2104,8.299,844,72.571
untuk,2100,8.283,816,70.163
dan,2069,8.161,827,71.109
saya,2050,8.086,817,70.249
ini,2043,8.059,821,70.593
tidak,1937,7.640,829,71.281
admin,706,2.785,324,27.859
bagus,625,2.465,261,22.442
produk,620,2.446,259,22.270
ukuran,605,2.386,221,19.003



Top 2-gram


,kemunculan,pct,dokumen,pct_dokumen
ini untuk,190,0.785,168,14.445
saya saya,185,0.765,153,13.156
sudah dan,184,0.761,168,14.445
dan sudah,181,0.748,161,13.844
sudah untuk,176,0.728,157,13.500
ini tidak,173,0.715,159,13.672
sudah saya,173,0.715,148,12.726
dan ini,173,0.715,160,13.758
untuk saya,171,0.707,147,12.640
saya dan,171,0.707,150,12.898



Top 3-gram


,kemunculan,pct,dokumen,pct_dokumen
contoh id review,126,0.547,126,10.834
https contoh id,126,0.547,126,10.834
admin https contoh,126,0.547,126,10.834
produk bagus kualitas,39,0.169,39,3.353
bagus kualitas baik,33,0.143,33,2.837
dan saya saya,31,0.135,31,2.666
dan sudah untuk,22,0.096,22,1.892
sudah saya dan,21,0.091,20,1.720
saya dan dan,20,0.087,20,1.720
ini untuk ini,20,0.087,20,1.720



Ringkasan n-gram


,total,unik,hapax
n,,,
1,25352,35,0
2,24189,490,4
3,23026,6105,844


**Angka cepat · kata/token yang perlu ditinjau**

Sampel. Token inspeksi memakai regex \w sehingga angka/underscore ikut. Kandidat pemeriksaan, bukan detektor typo. Non-ASCII dapat valid. KNOWN_WORDS hanya dipakai bila diisi.

In [35]:
raw_counts = pd.Series(Counter(w for t in sample_text.str.casefold() for w in re.findall(r"\b\w+\b",t)),dtype="int64").sort_values(ascending=False)
terms = raw_counts.index.to_series().astype("string")
odd_flags = pd.DataFrame({"panjang>=20":terms.str.len().ge(20),"huruf_ulang>=3":terms.str.count(r"([^\W\d_])\1{2,}").gt(0),
    "huruf+angka":terms.str.contains(r"[a-z]") & terms.str.contains(r"\d"),
    "konsonan>=6":terms.str.contains(r"[bcdfghjklmnpqrstvwxyz]{6,}"),"non_ASCII":terms.str.contains(r"[^\x00-\x7f]")})
if KNOWN_WORDS: odd_flags["di_luar_referensi"] = ~terms.isin({w.casefold() for w in KNOWN_WORDS})
show("Flag kandidat kata",pd.DataFrame({"tipe_kata":odd_flags.sum(),"kemunculan":odd_flags.mul(raw_counts,axis=0).sum(),
    "%_token":odd_flags.mul(raw_counts,axis=0).sum()/(raw_counts.sum() or np.nan)*100}))
for name in odd_flags: show(name,raw_counts[odd_flags[name]].head(TOP).rename("kemunculan").to_frame())


Flag kandidat kata

,tipe_kata,kemunculan,%_token
panjang>=20,0,0,0.0
huruf_ulang>=3,0,0,0.0
huruf+angka,0,0,0.0
konsonan>=6,0,0,0.0
non_ASCII,0,0,0.0



panjang>=20


,kemunculan



huruf_ulang>=3


,kemunculan


... (2 outputs left out)


konsonan>=6


,kemunculan



non_ASCII


,kemunculan


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [36]:
if RUN_PLOTS:
    from sklearn.feature_extraction.text import CountVectorizer

    def ngram_counts(texts, n=1):
        vectorizer = CountVectorizer(ngram_range=(n, n), stop_words=sorted(STOPWORDS) or None,
                                     max_features=20_000, token_pattern=r"(?u)\b[^\W\d_]{2,}\b")
        try:
            matrix = vectorizer.fit_transform(texts)
        except ValueError as error:
            if "empty vocabulary" in str(error): return pd.Series(dtype=float)
            raise
        return pd.Series(np.asarray(matrix.sum(axis=0)).ravel(), index=vectorizer.get_feature_names_out())

    for n in (1, 2, 3):
        bars(ngram_counts(sample_text, n), f"Top {n}-gram · sampel {len(sample_text):,} teks")

## 16 · Kurva Zipf

Pakai `sample_text`. Menghitung semua kata pada sampel tanpa potongan `max_features`; sumbu log membantu membaca ekor panjang kosakata.

**Angka cepat · kosakata, hapax, kata langka**

Sampel. Hapax=tepat 1 kemunculan, dis-legomena=2. % tipe memakai jumlah kata unik; % token memakai semua kemunculan. TTR dipengaruhi ukuran sampel.

In [37]:
show("Ringkasan kosakata",pd.Series({"dokumen_sampel":len(word_lists),"dokumen_tanpa_token_huruf":word_lists.map(len).eq(0).sum(),
    "token":vocab_counts.sum(),"tipe_unik":len(vocab_counts),"TTR_%":pct(len(vocab_counts),vocab_counts.sum()),
    "hapax_tipe":vocab_counts.eq(1).sum(),"hapax_%_tipe":pct(vocab_counts.eq(1).sum(),len(vocab_counts)),
    "hapax_%_token":pct(vocab_counts.eq(1).sum(),vocab_counts.sum()),"dis_legomena_tipe":vocab_counts.eq(2).sum(),
    "dis_legomena_%_tipe":pct(vocab_counts.eq(2).sum(),len(vocab_counts)),"dis_legomena_%_token":pct(vocab_counts[vocab_counts.eq(2)].sum(),vocab_counts.sum()),
    "langka_<=5_%_tipe":pct(vocab_counts.le(5).sum(),len(vocab_counts)),
    "langka_<=5_%_token":pct(vocab_counts[vocab_counts.le(5)].sum(),vocab_counts.sum()),
    "top10_%_token":pct(vocab_counts.head(10).sum(),vocab_counts.sum())}))
show("Contoh hapax",word_table.loc[vocab_counts.eq(1)].head(TOP))
show("Distribusi frekuensi tipe kata",distribution(vocab_counts).sort_index().rename_axis("kemunculan_per_kata").head(TOP))


Ringkasan kosakata


,0
dokumen_sampel,1163.000
dokumen_tanpa_token_huruf,0.000
token,25352.000
tipe_unik,35.000
TTR_%,0.138
hapax_tipe,0.000
hapax_%_tipe,0.000
hapax_%_token,0.000
dis_legomena_tipe,0.000
dis_legomena_%_tipe,0.000



Contoh hapax


,kemunculan,%_token,dokumen,%_dokumen



Distribusi frekuensi tipe kata


,n,%
kemunculan_per_kata,,
33,1,2.857
126,4,11.429
427,1,2.857
428,1,2.857
432,1,2.857
444,1,2.857
448,1,2.857
451,1,2.857
456,1,2.857


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [38]:
if RUN_PLOTS:
    counts = Counter(word for value in sample_text.str.lower() for word in re.findall(r"[^\W\d_]+", value))
    frequency = np.array(sorted(counts.values(), reverse=True))
    if len(frequency):
        plt.figure()
        plt.loglog(np.arange(1, len(frequency) + 1), frequency, color=BLUE, lw=2)
        finish("Frekuensi kosakata · sampel", "Peringkat kata (log)", "Frekuensi (log)")
    else:
        print("Lewati: tidak ada kata.")

## 16a · Bahasa · angka saja
Deteksi dokumen memakai [langid.py](https://github.com/saffsd/langid.py), model bawaan paket tanpa download model saat run. Bila belum terpasang, jalankan `%pip install langid threadpoolctl` sekali pada kernel ini.

Maksimal LANG_SAMPLE baris terisi diambil acak dari sampel kosakata. Hasil tidak diekstrapolasi menjadi jumlah seluruh dataset. Kurang LANG_MIN_WORDS atau skor normalisasi < LANG_MIN_SCORE menjadi **tidak_pasti**. URL/mention dikeluarkan hanya dari input detektor; maksimal 3.000 karakter dipakai. Skor bukan jaminan akurasi; bahasa serumpun, teks pendek, campur kode perlu dibaca contohnya. Hasil satu bahasa dominan, bukan bahasa per kata.

In [39]:
language_text = sample_text.sample(min(LANG_SAMPLE,len(sample_text)),random_state=SEED)
language_input = language_text.str.replace(r"https?://\S+|www\.\S+|@\w+"," ",regex=True).str.slice(0,3000)
language_rows = pd.DataFrame({"teks":language_text,"bahasa":"belum_dideteksi","skor":np.nan})
try:
    from langid.langid import LanguageIdentifier,model
    from threadpoolctl import threadpool_limits
    detector = LanguageIdentifier.from_modelstring(model,norm_probs=True)
    eligible = language_input.str.findall(r"[^\W\d_]+").str.len().ge(LANG_MIN_WORDS)
    with threadpool_limits(limits=1):
        cache = {s:detector.classify(s) for s in language_input[eligible].unique()}
    predictions = language_input.map(lambda s:cache.get(s,("tidak_pasti",np.nan)))
    language_rows["bahasa"] = predictions.map(lambda p:p[0]).astype("string")
    language_rows["skor"] = pd.to_numeric(predictions.map(lambda p:p[1]),errors="coerce").astype(float)
    language_rows.loc[language_rows.skor.lt(LANG_MIN_SCORE),"bahasa"] = "tidak_pasti"
except ImportError: print("Detektor belum terpasang: %pip install langid threadpoolctl")

In [40]:
print(f"Bahasa: {len(language_rows):,} baris sampel dari {filled_text.sum():,} teks terisi.")
show("Estimasi bahasa · % sampel bahasa",distribution(language_rows.bahasa))
show("Skor dan contoh",language_rows.groupby("bahasa").agg(n=("teks","size"),skor_median=("skor","median"),
    contoh=("teks",lambda s:s.iloc[0][:160])).sort_values("n",ascending=False))
if has(SOURCE): show("Bahasa x sumber · % sampel tiap sumber",pd.crosstab(df.loc[language_rows.index,SOURCE],language_rows.bahasa,normalize="index")*100)

Bahasa: 500 baris sampel dari 1,163 teks terisi.

Estimasi bahasa · % sampel bahasa


,n,%
bahasa,,
id,423,84.6
tidak_pasti,75,15.0
tr,1,0.2
ms,1,0.2



Skor dan contoh


,n,skor_median,contoh
bahasa,,,
id,423,0.999,ukuran sudah tidak sudah tidak saya sudah dan dan dan bagus tidak bahan dan tidak kualitas rusak sudah bahan dan saya tidak saya ukuran ...
tidak_pasti,75,0.663,admin dan dan bantuan admin admin saya bantuan bantuan
ms,1,0.834,ini murah diskon dan promo ini mahal mahal saya promo
tr,1,0.960,lambat kurir kirim ini kurir pengiriman ini saya



Bahasa x sumber · % sampel tiap sumber


bahasa,id,ms,tidak_pasti,tr
source,,,,
aplikasi,86.312,0.38,13.308,0.000
sosmed,81.955,0.00,18.045,0.000
web,83.654,0.00,15.385,0.962


**Penanda bahasa daerah & kemungkinan campur bahasa**
Kamus kecil di bawah hanya indikator kata, bukan detektor bahasa/proporsi penutur. Minimal dua kata penanda unik untuk satu flag; flag dapat overlap. Tidak ada hit tidak berarti pasti Indonesia. Bahasa yang tidak tercakup kamus tidak dapat dinilai.

In [41]:
MARKERS = {"id":"yang tidak bukan belum untuk dengan karena sudah sangat","en":"the this that with very because would should really",
    "jv":"ora mboten kowe piye mangan durung ngono ngene saiki","su":"teu henteu abdi kumaha naon ngeunah atuh euy ayeuna",
    "min":"indak rancak lamak bana sangaik harago cukuik labiah"}
marker_flags = pd.DataFrame({k:word_lists.map(lambda seq:len(set(seq)&set(words.split()))>=2).astype(bool) for k,words in MARKERS.items()})
show("Penanda bahasa · % sampel kosakata",flag_rates(marker_flags))
print("Kandidat >=2 kelompok penanda (% sampel):",marker_flags.sum(axis=1).ge(2).mean()*100)
show("Contoh kandidat campur bahasa",sample_text[marker_flags.sum(axis=1).ge(2)].head(TOP).str.slice(0,180).to_frame("teks"))


Penanda bahasa · % sampel kosakata


,n,%
id,870,74.807
en,0,0.000
jv,0,0.000
su,0,0.000
min,0,0.000


Kandidat >=2 kelompok penanda (% sampel): 0.0

Contoh kandidat campur bahasa


,teks


## 17 · Pola noise teks

Pakai `text`. Persentase seluruh baris; satu teks bisa mengandung beberapa pola. Ini indikator untuk diperiksa, bukan alasan otomatis menghapus.

**Angka cepat · jumlah & persen setiap noise**

Seluruh baris. Dua penyebut: semua baris dan teks terisi. Flag tumpang tindih; negasi, angka, slang, non-ASCII juga dapat menjadi sinyal penting.

In [42]:
if has(TEXT):
    report = flag_rates(noise_flags).rename(columns={"%":"%_semua_baris"})
    report["n_teks_terisi"] = noise_flags.loc[filled_text].sum()
    report["%_teks_terisi"] = noise_flags.loc[filled_text].mean()*100
    show("Seluruh indikator teks",report)
    show("Jumlah flag per baris · % seluruh data",distribution(noise_flags.sum(axis=1)).sort_index())
    show("Contoh per indikator",pd.Series({name:text[noise_flags[name]].head(2).str.slice(0,140).tolist() for name in noise_flags}))


Seluruh indikator teks


,n,%_semua_baris,n_teks_terisi,%_teks_terisi
negasi,829,69.083,829,71.281
kata ulang berurutan,798,66.5,798,68.616
hashtag,126,10.5,126,10.834
baca berulang,126,10.5,126,10.834
URL,126,10.5,126,10.834
mention,126,10.5,126,10.834
huruf+angka,0,0.0,0,0.0
angka,0,0.0,0,0.0
email,0,0.0,0,0.0
emoji/simbol,0,0.0,0,0.0



Jumlah flag per baris · % seluruh data


,n,%
0,186,15.5
1,326,27.167
2,562,46.833
4,16,1.333
5,43,3.583
6,67,5.583



Contoh per indikator


,0
URL,"[ini sudah bagus produk kualitas produk @admin!!! https://contoh.id #review, sudah bahan sudah produk tidak ini @admin!!! https://contoh..."
email,[]
mention,"[ini sudah bagus produk kualitas produk @admin!!! https://contoh.id #review, sudah bahan sudah produk tidak ini @admin!!! https://contoh..."
hashtag,"[ini sudah bagus produk kualitas produk @admin!!! https://contoh.id #review, sudah bahan sudah produk tidak ini @admin!!! https://contoh..."
angka,[]
huruf+angka,[]
emoji/simbol,[]
non-ASCII,[]
huruf berulang >=3,[]
baca berulang,"[ini sudah bagus produk kualitas produk @admin!!! https://contoh.id #review, sudah bahan sudah produk tidak ini @admin!!! https://contoh..."


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [43]:
if RUN_PLOTS:
    PATTERNS = {"URL": r"https?://|www\.", "Mention": r"@\w+", "Hashtag": r"#\w+",
                "Huruf berulang": r"([a-zA-Z])\1{2,}",
                "Angka": r"\d", "HTML": r"<[^>]+>", "Tanda baca berulang": r"[!?\.]{2,}"}
    if has(TEXT):
        rates = pd.Series({name: text.str.count(pattern).gt(0).mean() * 100 for name, pattern in PATTERNS.items()})
        bars(rates, "Pola teks", "% baris", YELLOW)

## 18 · Duplikat teks

Pakai `text`. Bandingkan baris ekstra duplikat persis dan setelah lowercase + rapikan spasi. Teks kosong dikeluarkan; bukan deteksi kesamaan semantik.

**Angka cepat · duplikat baris & teks**

Ekstra=setelah kemunculan pertama; semua anggota=termasuk pertama. Normalisasi hanya casefold+spasi. Teks kosong dikeluarkan dari hitungan duplikat teks.

In [44]:
try: row_duplicates = df.duplicated()
except TypeError: row_duplicates = df.map(repr).duplicated()  # kolom list/dict
print(f"Duplikat baris lengkap ekstra={row_duplicates.sum():,} ({row_duplicates.mean():.2%})")
dup_summary = []
for name,values in {"persis":text[filled_text],"casefold+spasi":normalized_text[filled_text]}.items():
    counts = values.value_counts()
    dup_summary.append({"jenis":name,"n_teks":len(values),"unik":len(counts),"ekstra":values.duplicated().sum(),
        "%_ekstra":values.duplicated().mean()*100,"semua_anggota_dup":values.duplicated(keep=False).sum(),
        "grup_dup":counts.gt(1).sum(),"grup_terbesar":counts.max()})
show("Duplikat teks",pd.DataFrame(dup_summary).set_index("jenis"))
show("Teks berulang",normalized_text[filled_text].value_counts().loc[lambda s:s.gt(1)].head(TOP).rename("n").to_frame())

Duplikat baris lengkap ekstra=0 (0.00%)

Duplikat teks


,n_teks,unik,ekstra,%_ekstra,semua_anggota_dup,grup_dup,grup_terbesar
jenis,,,,,,,
persis,1163,1131,32,2.752,33,1,33
casefold+spasi,1163,1131,32,2.752,33,1,33



Teks berulang


,n
text,
produk bagus kualitas baik,33


**Angka cepat · konflik label**

Klasifikasi satu label; teks dan label terisi. Casefold+spasi. Konflik perlu ditinjau bersama konteks, tidak otomatis diganti ke label mayoritas.

In [45]:
if KIND=="category" and labels is not None:
    audit = pd.DataFrame({"teks":normalized_text,"label":labels}).loc[filled_text & labels.notna()]
    conflicts = audit.groupby("teks").label.nunique().loc[lambda s:s.gt(1)]
    mask = audit.teks.isin(conflicts.index)
    print(f"Grup konflik={len(conflicts):,}; baris terlibat={mask.sum():,} ({pct(mask.sum(),len(audit)):.2f}% baris berlabel+teks terisi)")
    show("Contoh konflik",audit[mask].groupby("teks").label.agg(n="size",labels=lambda s:s.value_counts().to_dict()).head(TOP))

Grup konflik=1; baris terlibat=31 (2.83% baris berlabel+teks terisi)

Contoh konflik


,n,labels
teks,,
produk bagus kualitas baik,31,"{'positif': 19, 'netral': 7, 'negatif': 5}"


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [46]:
if RUN_PLOTS:
    valid = text[text.str.strip().ne("")]
    normalized = valid.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    if len(valid):
        rates = pd.Series({"Persis": valid.duplicated().mean() * 100,
                           "Lowercase + spasi": normalized.duplicated().mean() * 100})
        bars(rates, "Baris ekstra duplikat · teks terisi", "% teks terisi", YELLOW)
        bars(normalized.value_counts().head(TOP), "Teks paling sering berulang")

## 19 · Train vs test

Pakai `test`, `TEXT`. ECDF jumlah kata tanpa bergantung pada label test; teks kosong tetap dihitung.

**Angka cepat · profil train vs test**

Seluruh baris masing-masing split. Persen duplikat hanya teks terisi. Tidak menguji pergeseran makna.

In [47]:
if has(TEXT) and test is not None and TEXT in test:
    split_stats = []
    for name,frame in [("train",df),("test",test)]:
        values = frame[TEXT].astype("string").fillna("")
        valid = values.str.strip().ne(""); length = values.str.split().str.len()
        split_stats.append({"split":name,"baris":len(frame),"NaN_%":frame[TEXT].isna().mean()*100,"kosong_%":(~valid).mean()*100,
            "unik_terisi":values[valid].nunique(),"dup_ekstra_%_terisi":values[valid].duplicated().mean()*100,
            "kata_mean":length.mean(),"kata_p50":length.quantile(.5),"kata_p95":length.quantile(.95),"kata_max":length.max()})
    show("Profil split",pd.DataFrame(split_stats).set_index("split"))
else: print("Lewati: perlu test dengan kolom teks.")

Lewati: perlu test dengan kolom teks.


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [48]:
if RUN_PLOTS:
    if has(TEXT) and test is not None and TEXT in test:
        plt.figure()
        for frame, name, color in [(df, "train", BLUE), (test, "test", YELLOW)]:
            lengths = frame[TEXT].fillna("").astype(str).str.split().str.len()
            if len(lengths): sns.ecdfplot(x=lengths, label=name, color=color, linewidth=2)
        plt.legend()
        finish("Distribusi panjang train vs test", "Jumlah kata", "Proporsi kumulatif")
    else:
        print("Lewati: perlu test dengan kolom teks yang sama.")

## 20 · Pasangan teks

Pakai `TEXT`, `TEXT_B`, `SAMPLE`. Scatter panjang kedua sisi dan histogram Jaccard kata. Jaccard mengukur overlap literal, bukan makna.

**Angka cepat · pasangan teks**

Pasangan lengkap memakai sampel maksimal SAMPLE. Jaccard himpunan kata lowercase; kedua sisi kosong memberi NaN.

In [49]:
if has(TEXT) and has(TEXT_B):
    pairs = df[[TEXT,TEXT_B]].dropna().astype(str)
    print(f"Lengkap non-null={len(pairs):,}/{len(df):,}; sisi B null={df[TEXT_B].isna().sum():,}")
    pairs = pairs.sample(min(SAMPLE,len(pairs)),random_state=SEED)
    a,b = pairs.iloc[:,0],pairs.iloc[:,1]
    left,right = a.str.casefold().str.split().map(set),b.str.casefold().str.split().map(set)
    metrics = pd.DataFrame({"kata_A":a.str.split().str.len(),"kata_B":b.str.split().str.len(),
        "Jaccard":[len(x&y)/len(x|y) if x|y else np.nan for x,y in zip(left,right)]})
    show("Statistik pasangan sampel",metrics.describe(percentiles=[.5,.9,.95,.99]).T)
    print("Identik persis (%):",a.eq(b).mean()*100,"; salah satu sisi kosong (%):",(a.str.strip().eq("")|b.str.strip().eq("")).mean()*100)
else: print("Lewati: isi TEXT_B.")

Lewati: isi TEXT_B.


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [50]:
if RUN_PLOTS:
    if has(TEXT) and has(TEXT_B):
        pairs = df[[TEXT, TEXT_B]].dropna().astype(str)
        pairs = pairs.sample(min(SAMPLE, len(pairs)), random_state=SEED)
        a, b = pairs.iloc[:, 0].str.split(), pairs.iloc[:, 1].str.split()
        sets_a = pairs.iloc[:, 0].str.lower().str.split().map(set)
        sets_b = pairs.iloc[:, 1].str.lower().str.split().map(set)
        overlap = [len(x & y) / max(1, len(x | y)) for x, y in zip(sets_a, sets_b)]
        if len(pairs):
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].scatter(a.str.len(), b.str.len(), color=BLUE, alpha=.3, s=15)
            axes[0].set(xlabel="Kata teks A", ylabel="Kata teks B", title="Panjang pasangan")
            axes[1].hist(overlap, bins=25, color=YELLOW, edgecolor="white")
            axes[1].set(xlabel="Jaccard kata (0–1)", ylabel="Jumlah", title="Overlap literal")
            plt.tight_layout(); plt.show(); plt.close()
    else:
        print("Lewati: isi TEXT_B untuk pasangan teks.")

## 21 · Multilabel

Pakai `targets` berupa list kolom biner 0/1. Nilai kosong dikeluarkan per label; ko-occurence memakai baris lengkap saja.

**Angka cepat · multilabel**

TARGET=list kolom biner. Kosong tetap unknown. Persen aktif memakai nilai terisi per label; ko-occurence memakai baris lengkap.

In [51]:
if len(targets)>1:
    y = df[targets].replace(LABEL_DROP,np.nan).apply(pd.to_numeric,errors="raise")
    if not y.stack().isin([0,1]).all(): raise ValueError("Multilabel perlu 0/1 atau kosong.")
    show("Profil label",pd.DataFrame({"terisi":y.notna().sum(),"missing":y.isna().sum(),"aktif":y.eq(1).sum(),"%_aktif_terisi":y.mean()*100}))
    complete = y.dropna()
    print(f"Label lengkap={len(complete):,}/{len(y):,}")
    show("Ko-occurence jumlah baris lengkap",complete.T@complete)
else: print("Lewati: TARGET bukan beberapa kolom biner.")

Lewati: TARGET bukan beberapa kolom biner.


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [52]:
if RUN_PLOTS:
    if len(targets) > 1:
        y = df[targets].replace(LABEL_DROP, np.nan).apply(pd.to_numeric, errors="raise")
        if not y.stack().isin([0, 1]).all(): raise ValueError("Target multilabel harus 0/1 atau kosong.")
        bars(y.mean().mul(100), "Prevalensi label · nilai terisi", "% aktif")
        complete = y.dropna()
        if len(complete):
            heat(complete.T @ complete, "Ko-occurence label · jumlah baris lengkap", True)
    else:
        print("Lewati: TARGET harus list beberapa kolom biner.")

## 22 · Aspek domain

Pakai `text`, `ASPECTS`. Cocok regex, bukan klasifikasi model. Isi kamus sesuai domain; heatmap berisi jumlah teks yang cocok kedua aspek.

**Angka cepat · aspek domain**

ASPECTS=regex buatan pengguna. Persen seluruh baris; satu teks bisa cocok beberapa aspek.

In [53]:
if has(TEXT) and ASPECTS:
    aspects = pd.DataFrame({k:text.str.contains(v,case=False) for k,v in ASPECTS.items()})
    show("Penyebutan aspek",flag_rates(aspects))
    show("Jumlah aspek per teks",distribution(aspects.sum(axis=1)).sort_index())
    show("Ko-occurence jumlah baris",aspects.astype(int).T@aspects.astype(int))
else: print("Lewati: isi ASPECTS.")

Lewati: isi ASPECTS.


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [54]:
if RUN_PLOTS:
    if has(TEXT) and ASPECTS:
        aspects = pd.DataFrame({name: text.str.contains(pattern, case=False) for name, pattern in ASPECTS.items()})
        bars(aspects.mean().mul(100), "Penyebutan aspek", "% seluruh baris")
        binary = aspects.astype(int)
        heat(binary.T @ binary, "Ko-occurence aspek · jumlah baris", True)
    else:
        print("Lewati: isi ASPECTS untuk domain dataset.")

## 23 · Anotasi BIO

Pakai `TAGS`. Hanya tag `B-` dihitung sebagai awal entitas. Histogram menampilkan panjang urutan tag, bukan panjang entitas atau validasi format BIO.

**Angka cepat · jumlah & validitas BIO**

O/B-/I-. Transisi invalid=I-X tidak didahului B-X/I-X. Tidak membuktikan alignment ke tokenizer model.

In [55]:
if has(TAGS):
    seqs = df[TAGS].dropna().map(lambda v:v.split() if isinstance(v,str) else list(v))
    count = pd.Series(Counter(tag for seq in seqs for tag in seq),dtype="int64").sort_values(ascending=False)
    invalid = seqs.map(lambda seq:any(not re.fullmatch(r"O|[BI]-.+",str(tag)) or
        (str(tag).startswith("I-") and (str(prev)=="O" or str(prev)[2:]!=str(tag)[2:])) for prev,tag in zip(["O"]+seq[:-1],seq)))
    print(f"Tag null={df[TAGS].isna().sum():,}; urutan invalid={invalid.sum():,} ({invalid.mean():.2%} urutan terisi)")
    show("Tag",count.rename("n").to_frame().assign(pct=count/(count.sum() or np.nan)*100).head(TOP))
    show("Panjang urutan",seqs.map(len).describe(percentiles=[.5,.95,.99]))
else: print("Lewati: isi TAGS.")

Lewati: isi TAGS.


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [56]:
if RUN_PLOTS:
    if has(TAGS):
        sequences = df[TAGS].dropna().map(lambda value: value.split() if isinstance(value, str) else list(value))
        entities = Counter(tag[2:] for seq in sequences for tag in seq if isinstance(tag, str) and tag.startswith("B-"))
        bars(pd.Series(entities, dtype=float), "Awal entitas per tipe")
        if len(sequences):
            plt.figure()
            sns.histplot(sequences.map(len), bins=30, color=BLUE)
            finish("Panjang urutan BIO", "Jumlah tag", "Jumlah baris")
    else:
        print("Lewati: isi TAGS untuk anotasi token.")

## 24 · Pareto kategori

Pakai `df`. Bar menunjukkan volume kategori, garis menunjukkan kontribusi kumulatif. Garis 80% membantu melihat dominasi; sumbu kanan selalu 0–100%.

**Angka cepat · kontribusi kategori**

Seluruh baris termasuk NaN. TOP hanya membatasi tampilan.

In [57]:
COL = SOURCE
if has(COL):
    table = distribution(df[COL]); table["kumulatif_%"] = table["%"].cumsum()
    show(f"Pareto {COL}",table.head(TOP))
    print("Kategori untuk mencapai >=80%:",int(np.searchsorted(table["kumulatif_%"].to_numpy(),80))+1)


Pareto source


,n,%,kumulatif_%
source,,,
aplikasi,621,51.750,51.750
sosmed,358,29.833,81.583
web,221,18.417,100.000


Kategori untuk mencapai >=80%: 2


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [58]:
if RUN_PLOTS:
    COL = SOURCE
    if has(COL):
        counts = df[COL].astype("string").fillna("(kosong)").value_counts()
        shown = counts.head(TOP).copy()
        if len(counts) > TOP: shown.loc["Lainnya (gabungan)"] = counts.iloc[TOP:].sum()
        fig, ax = plt.subplots(figsize=(11, 4.5))
        ax.bar(range(len(shown)), shown, color=BLUE, width=.65)
        ax.set_xticks(range(len(shown)), shown.index, rotation=35, ha="right")
        right = ax.twinx()
        right.plot(range(len(shown)), shown.cumsum()/shown.sum()*100, color=YELLOW, marker="o", lw=3)
        right.axhline(80, color=INK, linestyle="--", alpha=.4)
        right.set(ylim=(0, 105), ylabel="Kontribusi kumulatif (%)")
        right.grid(False)
        plt.sca(ax)
        finish(f"Dominasi kategori · {COL}", "", "Jumlah baris")
    else:
        print("Lewati: kolom kategori belum tersedia.")

## 25 · Stacked bar 100%

Pakai `labels`, `SOURCE`. Bandingkan komposisi label sambil melihat jumlah baris berlabel per sumber. Netral tambahan dipakai jika kategori lebih dari dua.

**Angka cepat · cakupan label per sumber**

Seluruh sumber termasuk NaN. Persen berlabel memakai seluruh baris masing-masing sumber.

In [59]:
if labels is not None and has(SOURCE):
    table = labels.notna().groupby(df[SOURCE],dropna=False).agg(n="size",berlabel="sum",proporsi="mean")
    show("Cakupan label",table.assign(pct_berlabel=table.proporsi*100).drop(columns="proporsi").sort_values("n",ascending=False).head(TOP))
    if KIND=="category": show("Label per sumber (%)",pd.crosstab(df[SOURCE],labels,normalize="index").mul(100).iloc[:TOP,:TOP])


Cakupan label


,n,berlabel,pct_berlabel
source,,,
aplikasi,621,589,94.847
sosmed,358,332,92.737
web,221,208,94.118



Label per sumber (%)


label,negatif,netral,positif
source,,,
aplikasi,19.525,30.051,50.424
sosmed,20.783,30.422,48.795
web,15.865,27.885,56.250


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [60]:
if RUN_PLOTS:
    if labels is not None and has(SOURCE):
        table = pd.crosstab(df[SOURCE], labels)
        if not table.empty:
            table = table.loc[table.sum(axis=1).nlargest(min(TOP, 10)).index]
            top = table.sum().nlargest(5).index
            shown = table[top].copy()
            if len(table.columns) > 5: shown["Lainnya (gabungan)"] = table.drop(columns=top).sum(axis=1)
            colors = [BLUE, YELLOW, INK, "#60758a", "#b4c1cb", "#e3e8ec"]
            ax = shown.div(table.sum(axis=1), axis=0).mul(100).plot.barh(stacked=True, color=colors, width=.6)
            for i, total in enumerate(table.sum(axis=1)): ax.text(101, i, f"n={total:,}", va="center", fontsize=9)
            ax.set_xlim(0, 120)
            ax.legend(loc="upper center", bbox_to_anchor=(.5, -.17), ncol=3, frameon=False)
            finish("Komposisi target per sumber", "% baris berlabel", "")
    else:
        print("Lewati: perlu label dan sumber.")

## 26 · Ridgeline panjang teks

Pakai `features`, `labels` atau `SOURCE`. Histogram densitas per kelompok dalam panel bertumpuk; garis kuning = median. Semua panel memakai sumbu X yang sama.

**Angka cepat · kuantil per kelompok**

Seluruh baris termasuk panjang 0. TOP kelompok terbesar.

In [61]:
COL = SOURCE
if has(TEXT) and has(COL):
    show("Kuantil panjang kata",features.kata.groupby(df[COL],dropna=False).describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]).sort_values("count",ascending=False).head(TOP))


Kuantil panjang kata


,count,mean,std,min,10%,25%,50%,75%,90%,95%,99%,max
source,,,,,,,,,,,,
aplikasi,621.0,20.374,17.608,0.0,5.0,9.00,16.0,27.00,39.0,52.0,74.20,140.0
sosmed,358.0,21.765,18.638,0.0,4.0,9.25,16.0,28.75,45.0,59.0,89.15,121.0
web,221.0,21.068,19.088,0.0,5.0,9.00,16.0,25.00,40.0,57.0,100.20,140.0


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [62]:
if RUN_PLOTS:
    COL = SOURCE
    if has(TEXT) and has(COL):
        groups = df[COL].value_counts().head(6).index
        if len(groups):
            fig, axes = plt.subplots(len(groups), 1, figsize=(10, 1.3*len(groups)+1), sharex=True, squeeze=False)
            limit = max(1, features["kata"].max())
            bins = np.linspace(0, limit+1, 35)
            for ax, group in zip(axes.flat, groups):
                values = features.loc[df[COL].eq(group), "kata"]
                ax.hist(values, bins=bins, density=True, color=BLUE, alpha=.9, edgecolor="white")
                ax.axvline(values.median(), color=YELLOW, lw=3)
                ax.set_ylabel(str(group), rotation=0, ha="right", va="center")
                ax.set_yticks([]); ax.grid(False)
            axes.flat[0].set_title("Bentuk distribusi panjang · garis kuning = median")
            axes.flat[-1].set_xlabel("Jumlah kata")
            plt.tight_layout(); plt.show(); plt.close()
    else:
        print("Lewati: perlu teks dan kelompok.")

## 27 · Peta kepadatan panjang

Pakai `features`. Hexbin menghindari tumpang tindih titik pada dataset besar. Skala warna log menunjukkan jumlah baris dalam tiap sel.

**Angka cepat · relasi karakter/kata**

Rasio hanya pada jumlah kata >0. Contoh teks panjang ada di 09.

In [63]:
if has(TEXT):
    show("Korelasi karakter/kata",features.corr(method="spearman"))
    show("Karakter per kata/spasi",features.loc[features.kata.gt(0),"karakter"].div(features.loc[features.kata.gt(0),"kata"]).describe(percentiles=[.5,.9,.95,.99]))


Korelasi karakter/kata


,karakter,kata
karakter,1.000,0.993
kata,0.993,1.000



Karakter per kata/spasi


,0
count,1163.0
mean,6.0
std,0.591
min,4.333
50%,5.905
90%,6.667
95%,7.076
99%,8.116
max,9.8


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [64]:
if RUN_PLOTS:
    if has(TEXT):
        plt.figure(figsize=(9, 5.5))
        image = plt.hexbin(features["kata"], features["karakter"], gridsize=32,
                           mincnt=1, bins="log", cmap=CMAP, linewidths=.3)
        plt.colorbar(image, label="Jumlah baris (skala log)", shrink=.8)
        finish("Kepadatan struktur teks", "Jumlah kata", "Jumlah karakter")

## 28 · Bubble kompleksitas teks

Pakai `lexical`. Maksimal `SAMPLE` titik. Warna = proporsi kata unik; ukuran = jumlah karakter. Rasio kata unik sangat dipengaruhi panjang teks.

**Angka cepat · keragaman per dokumen**

Split spasi. Teks kosong dikeluarkan dari ringkasan. Rasio unik dipengaruhi panjang teks.

In [65]:
if has(TEXT):
    show("Profil leksikal teks terisi",lexical.loc[filled_text].describe(percentiles=[.5,.9,.95,.99]).T)
    bins = pd.cut(features.kata,[-1,0,5,10,20,50,100,np.inf])
    show("Keragaman menurut panjang",lexical.groupby(bins,observed=True)["unik_%"].agg(n="size",median="median",mean="mean"))


Profil leksikal teks terisi


,count,mean,std,min,50%,90%,95%,99%,max
karakter,1163.0,129.33448,108.384193,9.0,100.0,252.8,342.9,571.16,867.0
kata,1163.0,21.582115,18.087065,2.0,17.0,42.0,56.0,93.9,140.0
unik_%,1163.0,56.386469,22.474218,8.571429,55.555556,88.888889,100.0,100.0,100.0
digit_%,1163.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kapital_%,1163.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Keragaman menurut panjang


,n,median,mean
kata,,,
"(-1.0, 0.0]",37,0.000,0.000
"(0.0, 5.0]",111,100.000,92.282
"(5.0, 10.0]",218,75.000,76.214
"(10.0, 20.0]",386,58.824,59.615
"(20.0, 50.0]",368,37.931,38.652
"(50.0, 100.0]",70,20.000,19.778
"(100.0, inf]",10,10.087,9.952


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [66]:
if RUN_PLOTS:
    if has(TEXT):
        view = lexical.loc[features["kata"].gt(0)].sample(min(SAMPLE, features["kata"].gt(0).sum()), random_state=SEED)
        if len(view):
            plt.figure(figsize=(10, 5))
            points = plt.scatter(view["kata"], view["unik_%"],
                      s=15+90*view["karakter"]/max(1,view["karakter"].max()),
                      c=view["unik_%"], cmap=CMAP, alpha=.65, edgecolors=BLUE, linewidths=.2)
            plt.colorbar(points, label="Kata unik (%)", shrink=.8)
            finish("Panjang vs keragaman leksikal", "Jumlah kata", "Kata unik / jumlah kata (%)")

## 29 · Evolusi komposisi label

Pakai `dates`, `labels`. Area ditumpuk menunjukkan volume, bukan persentase. Maksimal lima label teratas + gabungan sisanya. Perubahan bisa berasal dari pergeseran pengambilan data.

**Angka cepat · label menurut waktu**

Klasifikasi; TOP periode terbaru/label terbesar. Persen dihitung sebelum pembatasan kolom.

In [67]:
if KIND=="category" and dates is not None and labels is not None:
    view = pd.DataFrame({"waktu":dates,"label":labels}).dropna()
    if len(view):
        table = view.groupby([pd.Grouper(key="waktu",freq=FREQ),"label"]).size().unstack(fill_value=0)
        cols = table.sum().nlargest(TOP).index
        show("Label per periode UTC",table[cols].tail(TOP))
        show("Komposisi per periode (%)",table.div(table.sum(axis=1),axis=0).mul(100)[cols].tail(TOP))


Label per periode UTC


label,positif,netral,negatif
waktu,,,
2025-03-30 00:00:00+00:00,28,13,6
2025-04-06 00:00:00+00:00,23,21,14
2025-04-13 00:00:00+00:00,24,22,10
2025-04-20 00:00:00+00:00,18,10,6
2025-04-27 00:00:00+00:00,14,8,7
2025-05-04 00:00:00+00:00,15,15,8
2025-05-11 00:00:00+00:00,23,8,4
2025-05-18 00:00:00+00:00,12,5,3
2025-05-25 00:00:00+00:00,11,5,7



Komposisi per periode (%)


label,positif,netral,negatif
waktu,,,
2025-03-30 00:00:00+00:00,59.574,27.660,12.766
2025-04-06 00:00:00+00:00,39.655,36.207,24.138
2025-04-13 00:00:00+00:00,42.857,39.286,17.857
2025-04-20 00:00:00+00:00,52.941,29.412,17.647
2025-04-27 00:00:00+00:00,48.276,27.586,24.138
2025-05-04 00:00:00+00:00,39.474,39.474,21.053
2025-05-11 00:00:00+00:00,65.714,22.857,11.429
2025-05-18 00:00:00+00:00,60.000,25.000,15.000
2025-05-25 00:00:00+00:00,47.826,21.739,30.435


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [68]:
if RUN_PLOTS:
    if dates is not None and labels is not None:
        view = pd.DataFrame({"waktu": dates, "label": labels}).dropna()
        if len(view):
            table = view.groupby([pd.Grouper(key="waktu", freq=FREQ), "label"]).size().unstack(fill_value=0)
            top = table.sum().nlargest(5).index
            shown = table[top].copy()
            if len(table.columns) > 5: shown["Lainnya (gabungan)"] = table.drop(columns=top).sum(axis=1)
            ax = shown.plot.area(color=[BLUE, YELLOW, INK, "#60758a", "#b4c1cb", "#e3e8ec"], alpha=.9, figsize=(11,5))
            ax.legend(loc="upper left", frameon=False, ncol=3)
            finish("Evolusi volume label (UTC)", "Waktu", "Jumlah baris berlabel")
    else:
        print("Lewati: perlu waktu dan label.")

## 30 · Konsentrasi grup / thread

Pakai `GROUP_COL`. Kurva menampilkan kontribusi kumulatif grup terbesar. Dominasi grup penting untuk memilih group split, tetapi chart ini tidak menguji validitas split.

**Angka cepat · konsentrasi grup**

Baris tanpa grup terpisah. Persen kontribusi memakai baris dengan grup terisi.

In [69]:
if has(GROUP_COL):
    count = df[GROUP_COL].value_counts()
    print(f"Grup unik={len(count):,}; baris tanpa grup={df[GROUP_COL].isna().sum():,}; grup 1 baris={count.eq(1).sum():,}")
    show("Ukuran grup",count.describe(percentiles=[.5,.9,.95,.99]))
    show("Grup terbesar",count.head(TOP).rename("n").to_frame().assign(pct=count.head(TOP)/(count.sum() or np.nan)*100))
    show("Kontribusi top-k grup (%)",pd.Series({k:pct(count.head(k).sum(),count.sum()) for k in [1,5,10,50,100]}))

Grup unik=38; baris tanpa grup=0; grup 1 baris=14

Ukuran grup


,count
count,38.000
mean,31.579
std,118.908
min,1.000
50%,2.500
90%,36.500
95%,97.400
99%,522.970
max,715.000



Grup terbesar


,n,pct
group,,
thread-1,715,59.583
thread-2,196,16.333
thread-3,80,6.667
thread-4,54,4.500
thread-6,29,2.417
thread-5,25,2.083
thread-7,14,1.167
thread-8,12,1.000
thread-21,6,0.500



Kontribusi top-k grup (%)


,0
1,59.583
5,89.500
10,94.750
50,100.000
100,100.000


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [70]:
if RUN_PLOTS:
    if has(GROUP_COL):
        counts = df[GROUP_COL].dropna().value_counts()
        if len(counts):
            x = np.r_[0, np.arange(1,len(counts)+1)/len(counts)]
            y = np.r_[0, counts.cumsum()/counts.sum()]
            plt.figure(figsize=(8,5))
            plt.fill_between(x, y, x, color=BLUE, alpha=.15)
            plt.plot(x, y, color=BLUE, lw=3, label="Grup terbesar lebih dulu")
            plt.plot([0,1],[0,1], color=YELLOW, lw=2, linestyle="--", label="Kontribusi merata")
            plt.gca().xaxis.set_major_formatter(PercentFormatter(1))
            plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
            plt.legend(frameon=False)
            finish("Konsentrasi baris menurut grup", "Proporsi grup", "Proporsi baris")
    else:
        print("Lewati: isi GROUP_COL untuk grup/thread.")

## 31 · Sidik noise per sumber

Pakai `text`, `SOURCE`. Setiap sel adalah persentase baris sumber yang cocok regex. Pola ditentukan lokal agar bisa dicopy tanpa sel noise sebelumnya.

**Angka cepat · noise per sumber**

Seluruh baris setiap sumber termasuk sumber missing. Persen indikator dapat overlap; n_baris memberi penyebutnya.

In [71]:
if has(TEXT) and has(SOURCE):
    group = noise_flags.groupby(df[SOURCE],dropna=False)
    table = group.mean().mul(100); table.insert(0,"n_baris",group.size())
    show("Indikator per sumber (%)",table.sort_values("n_baris",ascending=False).head(TOP))


Indikator per sumber (%)


,n_baris,URL,email,mention,hashtag,angka,huruf+angka,emoji/simbol,non-ASCII,huruf berulang >=3,baca berulang,KAPITAL >=3,HTML tag,HTML entity,indikasi encoding rusak,invisible/bidi,kontrol,newline/tab,spasi berulang,spasi tepi,negasi,slang contoh,kata ulang berurutan
source,,,,,,,,,,,,,,,,,,,,,,,
aplikasi,621,10.467,0.0,10.467,10.467,0.0,0.0,0.0,0.0,0.0,10.467,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,69.887,0.0,66.828
sosmed,358,9.777,0.0,9.777,9.777,0.0,0.0,0.0,0.0,0.0,9.777,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,68.436,0.0,67.039
web,221,11.765,0.0,11.765,11.765,0.0,0.0,0.0,0.0,0.0,11.765,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67.873,0.0,64.706


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [72]:
if RUN_PLOTS:
    patterns = {"URL": r"https?://|www\.", "Mention": r"@\w+", "Hashtag": r"#\w+",
                "Angka": r"\d", "Kapital": r"[A-Z]{3,}", "Baca ulang": r"[!?]{2,}"}
    if has(TEXT) and has(SOURCE):
        flags = pd.DataFrame({name: text.str.contains(regex) for name, regex in patterns.items()})
        table = flags.groupby(df[SOURCE]).mean().mul(100)
        order = df[SOURCE].value_counts().head(TOP).index
        heat(table.reindex(order), "Sidik noise per sumber (% baris)", True)
    else:
        print("Lewati: perlu teks dan sumber.")

## 32 · Overlap train–test

Pakai `test`, `TEXT`. Ukur persentase baris test yang teksnya muncul di train setelah lowercase + rapikan spasi. Tidak mendeteksi parafrasa; overlap bukan otomatis bukti pelanggaran.

**Angka cepat · overlap train-test**

Casefold+spasi. Teks kosong dikeluarkan. Overlap baris dan tipe unik dibedakan; tidak mendeteksi parafrasa.

In [73]:
if has(TEXT) and test is not None and TEXT in test:
    other = test[TEXT].astype("string").fillna("").str.casefold().str.replace(r"\s+"," ",regex=True).str.strip()
    a,b = normalized_text[filled_text],other[other.ne("")]
    overlap = set(a)&set(b)
    show("Overlap literal",pd.Series({"tipe_overlap":len(overlap),"train_baris_terisi":len(a),"test_baris_terisi":len(b),
        "test_baris_cocok_train":b.isin(overlap).sum(),"test_%_baris_terisi":b.isin(overlap).mean()*100,
        "test_%_tipe_unik":pct(len(overlap),b.nunique()),"train_%_baris_terisi":a.isin(overlap).mean()*100}))
    show("Contoh overlap",pd.Series(sorted(overlap)[:TOP],name="teks",dtype="string").to_frame())

**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [74]:
if RUN_PLOTS:
    if has(TEXT) and test is not None and TEXT in test:
        norm = lambda s: s.fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
        train_key, test_key = norm(df[TEXT]), norm(test[TEXT])
        valid = test_key[test_key.ne("")]
        if len(valid):
            overlap = valid.isin(set(train_key[train_key.ne("")])).sum()
            sizes = [overlap, len(valid)-overlap]
            fig, ax = plt.subplots(figsize=(7,5))
            ax.pie(sizes, colors=[YELLOW, BLUE], startangle=90,
                   wedgeprops={"width": .22, "edgecolor": "white", "linewidth": 3})
            ax.text(0,.02,f"{overlap/len(valid):.1%}",ha="center",fontsize=30,weight="bold")
            ax.text(0,-.22,"test cocok train",ha="center",fontsize=10)
            ax.legend([f"Overlap: {overlap:,}", f"Tidak overlap: {len(valid)-overlap:,}"], frameon=False, loc="lower center")
            finish("Overlap literal train–test · teks terisi")
    else:
        print("Lewati: perlu teks train dan test.")

## 33 · Kata pembeda dua kelompok

Pakai `sample_text`, `labels` atau sumber. Isi `A`, `B` sesuai nilai kategori. Butterfly menunjukkan selisih frekuensi kata per 1.000 token, bukan signifikansi statistik.

**Angka cepat · kata pembeda kelompok**

Sampel, tanpa STOPWORDS. A/B=None memilih dua kelompok terbesar. Frekuensi per 1.000 token tiap kelompok.

In [75]:
COL, A, B = SOURCE, None, None
if has(COL) and df[COL].nunique()>=2:
    groups = df[COL].value_counts().index
    a,b = groups[0] if A is None else A, groups[1] if B is None else B
    tables = []
    for group in [a,b]:
        seqs = word_lists[df.loc[word_lists.index,COL].eq(group)]
        count = pd.Series(Counter(w for seq in seqs for w in seq if w not in STOPWORDS),dtype="float64")
        tables.append(count/(count.sum() or np.nan)*1000)
    table = pd.concat(tables,axis=1,keys=["A_per1000","B_per1000"]).fillna(0)
    table["selisih_A_minus_B"] = table.A_per1000-table.B_per1000
    print("A:",a,"| B:",b)
    show("Pembeda terbesar",table.loc[table.selisih_A_minus_B.abs().nlargest(TOP).index])

A: aplikasi | B: sosmed

Pembeda terbesar


,A_per1000,B_per1000,selisih_A_minus_B
saya,110.765,98.918,11.848
harga,20.328,28.407,-8.079
bahan,31.840,24.518,7.322
kualitas,33.499,26.547,6.952
murah,22.713,28.238,-5.525
biaya,20.950,26.040,-5.090
cepat,27.484,32.465,-4.981
ramah,29.766,34.664,-4.898
bagus,33.810,29.084,4.727
promo,21.780,26.040,-4.260


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [76]:
if RUN_PLOTS:
    COL, A, B = SOURCE, None, None  # None = dua kelompok terbesar
    if has(COL):
        groups = df[COL].dropna().value_counts().index.tolist()
        if len(groups) >= 2:
            a, b = (groups[0] if A is None else A), (groups[1] if B is None else B)
            def frequency(group):
                values = sample_text[df.loc[sample_text.index, COL].eq(group)]
                count = Counter(w for t in values.str.lower() for w in re.findall(r"[^\W\d_]{2,}",t) if w not in STOPWORDS)
                return pd.Series(count, dtype=float).div(max(1,sum(count.values()))).mul(1000)
            left, right = frequency(a), frequency(b)
            diff = left.subtract(right, fill_value=0)
            chosen = pd.concat([diff.nlargest(8), diff.nsmallest(8)]).groupby(level=0).first().sort_values()
            if len(chosen):
                ax = chosen.plot.barh(color=[BLUE if x>=0 else YELLOW for x in chosen], figsize=(10,6))
                ax.axvline(0,color=INK,lw=1); ax.grid(axis="y",visible=False)
                finish(f"Kata pembeda · kiri: {b} | kanan: {a}", "Selisih frekuensi per 1.000 token", "")
    else:
        print("Lewati: perlu kolom kelompok.")

## 34 · Coverage kosakata

Pakai `sample_text`. Kurva kontribusi kumulatif semua token; titik kuning menunjukkan banyaknya kata unik untuk mencapai 90%. Tidak bergantung sel Zipf.

**Angka cepat · coverage kosakata**

Sampel. Tipe kata teratas untuk mencakup 50-100% token; vocabulary tidak dipotong saat menghitung.

In [77]:
if vocab_counts.sum():
    cumulative = vocab_counts.cumsum()/vocab_counts.sum()
    coverage = pd.Series({f"{goal:.0%} token":int(np.searchsorted(cumulative.to_numpy(),goal))+1 for goal in [.5,.8,.9,.95,.99,1.]})
    show("Kata unik yang dibutuhkan",coverage.rename("n_tipe").to_frame().assign(pct_vocab=coverage/len(vocab_counts)*100))


Kata unik yang dibutuhkan


,n_tipe,pct_vocab
50% token,7,20.000
80% token,20,57.143
90% token,26,74.286
95% token,29,82.857
99% token,33,94.286
100% token,35,100.000


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [78]:
if RUN_PLOTS:
    count = Counter(w for t in sample_text.str.lower() for w in re.findall(r"[^\W\d_]+", t))
    frequency = np.array(sorted(count.values(), reverse=True))
    if len(frequency):
        coverage = np.cumsum(frequency)/frequency.sum()
        k = int(np.searchsorted(coverage,.9))+1
        plt.figure(figsize=(10,4.5))
        plt.plot(np.arange(1,len(frequency)+1), coverage*100, color=BLUE, lw=2.5)
        plt.scatter([k],[coverage[k-1]*100], color=YELLOW, edgecolor=INK, s=130, zorder=3)
        plt.annotate(f"90% token: {k:,} kata unik", (k, coverage[k-1]*100),
                     xytext=(12,-32), textcoords="offset points", weight="bold")
        plt.axhline(90, color=INK, lw=1, alpha=.3, linestyle="--")
        finish("Cakupan kosakata · sampel", "Jumlah kata unik (peringkat frekuensi)", "Token tercakup (%)")

## 35 · Bubble kata × sumber

Pakai `sample_text`, `SOURCE`. Ukuran dan warna titik = jumlah kemunculan kata per 1.000 token sumber; maksimal 12 kata dan 6 sumber. Ini per-token, bukan persentase dokumen.

**Angka cepat · kosakata per sumber**

Sampel. Per 1.000 token huruf tiap sumber; stopword tetap dihitung dalam penyebut. Maksimal TOP sumber/kata.

In [79]:
if has(SOURCE):
    source_words,summary = {},[]
    for group in df[SOURCE].value_counts().head(TOP).index:
        seqs = word_lists[df.loc[word_lists.index,SOURCE].eq(group)]
        count = pd.Series(Counter(w for seq in seqs for w in seq),dtype="float64")
        source_words[group] = count/(count.sum() or np.nan)*1000
        summary.append({"sumber":group,"dok_sampel":len(seqs),"token":count.sum(),"unik":len(count),"hapax_%_tipe":pct(count.eq(1).sum(),len(count))})
    if summary:
        show("Kosakata sumber",pd.DataFrame(summary).set_index("sumber"))
        show("Kata x sumber per 1.000 token",pd.DataFrame(source_words).reindex(vocab_counts.head(TOP).index).fillna(0))


Kosakata sumber


,dok_sampel,token,unik,hapax_%_tipe
sumber,,,,
aplikasi,600,12782.0,35,0.0
sosmed,347,7862.0,35,0.0
web,216,4708.0,35,0.0



Kata x sumber per 1.000 token


,aplikasi,sosmed,web
sudah,83.320,82.040,83.687
untuk,83.399,82.167,82.413
dan,81.834,84.457,76.253
saya,83.555,74.409,84.325
ini,80.426,81.150,80.076
tidak,76.279,77.716,74.554
admin,26.913,29.509,27.613
bagus,25.505,21.877,26.975
produk,24.722,23.785,24.851
ukuran,25.348,22.132,22.727


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [80]:
if RUN_PLOTS:
    if has(SOURCE) and len(sample_text):
        groups = df[SOURCE].value_counts().head(6).index
        frequency = {}
        for group in groups:
            values = sample_text[df.loc[sample_text.index,SOURCE].eq(group)]
            count = Counter(w for t in values.str.lower() for w in re.findall(r"[^\W\d_]{2,}", t) if w not in STOPWORDS)
            frequency[str(group)] = pd.Series(count,dtype=float).div(max(1,sum(count.values()))).mul(1000)
        table = pd.DataFrame(frequency).fillna(0)
        if len(table):
            table = table.loc[table.sum(axis=1).nlargest(12).index]
            yy, xx = np.indices(table.shape)
            plt.figure(figsize=(9,6))
            points = plt.scatter(xx.ravel(), yy.ravel(), s=table.to_numpy().ravel()*5,
                                 c=table.to_numpy().ravel(), cmap=CMAP, edgecolor=BLUE, linewidth=.6)
            plt.xticks(range(len(table.columns)), table.columns)
            plt.yticks(range(len(table)), table.index)
            plt.gca().invert_yaxis(); plt.colorbar(points,label="Per 1.000 token",shrink=.7)
            finish("Profil kosakata per sumber", "Sumber", "")
    else:
        print("Lewati: perlu sumber dan teks terisi.")

## 36 · Jaringan aspek

Pakai `ASPECTS`, `text`. Tanpa networkx. Ukuran simpul = jumlah teks aspek; tebal edge = jumlah teks kedua aspek. Maksimal 10 aspek; layout lingkaran tidak menyatakan jarak semantik.

**Angka cepat · relasi antar-aspek**

Persen bersyarat: dari teks aspek baris, berapa persen juga cocok aspek kolom? Aspek tidak muncul memberi NaN.

In [81]:
if has(TEXT) and ASPECTS:
    flags = pd.DataFrame({k:text.str.contains(v,case=False) for k,v in ASPECTS.items()}).astype(int)
    co = flags.T@flags
    show("Ko-occurence aspek",co)
    show("Aspek kolom | aspek baris (%)",co.div(flags.sum().replace(0,np.nan),axis=0).mul(100))

**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [82]:
if RUN_PLOTS:
    if has(TEXT) and ASPECTS:
        flags = pd.DataFrame({k:text.str.contains(v,case=False) for k,v in ASPECTS.items()})
        names = flags.sum().nlargest(10).index
        flags = flags[names].astype(int); co = flags.T @ flags
        angle = np.linspace(0,2*np.pi,len(names),endpoint=False)
        xy = np.column_stack([np.cos(angle),np.sin(angle)])
        fig, ax = plt.subplots(figsize=(8,7))
        for i in range(len(names)):
            for j in range(i):
                weight = co.iloc[i,j]
                if weight: ax.plot(*xy[[i,j]].T,color=BLUE,alpha=.3,lw=.5+5*weight/max(1,co.to_numpy().max()))
        size = 200+1800*flags.sum()/max(1,flags.sum().max())
        ax.scatter(*xy.T,s=size,color=YELLOW,edgecolor=INK,zorder=3)
        for name,(x,y) in zip(names,xy): ax.text(x*1.24,y*1.24,name,ha="center",va="center",weight="bold")
        ax.set(xlim=(-1.65,1.65),ylim=(-1.5,1.5),aspect="equal"); ax.axis("off")
        finish("Jaringan penyebutan bersama · aspek")
    else:
        print("Lewati: isi ASPECTS sesuai domain.")

## 37 · Peta leksikal TF-IDF → SVD

Pakai `sample_text`, opsional `labels`. Membutuhkan scikit-learn. Maksimal `PROJECTION_N` dokumen, 8.000 fitur. Proyeksi 2D berbasis kata, bukan embedding semantik; warna hanya kategori label, bukan hasil clustering.

**Angka cepat · document frequency & IDF**

Sampel. Tanpa menjalankan proyeksi/SVD. Smoothed IDF = log((1+n_dok)/(1+df))+1; bukan skor kualitas model.

In [83]:
if len(vocab_counts):
    idf = np.log((1+len(word_lists))/(1+doc_counts))+1
    show("Document frequency",doc_counts.describe(percentiles=[.5,.9,.95,.99]))
    show("IDF smoothed",idf.describe(percentiles=[.5,.9,.95,.99]))
    show("Kata paling tersebar",word_table.sort_values("dokumen",ascending=False).head(TOP))


Document frequency


,0
count,35.000
mean,302.286
std,246.319
min,33.000
50%,209.000
90%,819.400
95%,827.600
99%,838.900
max,844.000



IDF smoothed


,0
count,35.000
mean,2.585
std,0.675
min,1.320
50%,2.713
90%,3.215
95%,3.215
99%,4.085
max,4.533



Kata paling tersebar


,kemunculan,%_token,dokumen,%_dokumen
sudah,2104,8.299,844,72.571
tidak,1937,7.640,829,71.281
dan,2069,8.161,827,71.109
ini,2043,8.059,821,70.593
saya,2050,8.086,817,70.249
untuk,2100,8.283,816,70.163
admin,706,2.785,324,27.859
bagus,625,2.465,261,22.442
produk,620,2.446,259,22.270
kualitas,594,2.343,243,20.894


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [84]:
if RUN_PLOTS:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    projection_text = sample_text.sample(min(PROJECTION_N,len(sample_text)),random_state=SEED)
    projection = None
    if len(projection_text) >= 3:
        vectorizer = TfidfVectorizer(max_features=8000,stop_words=sorted(STOPWORDS) or None)
        try:
            matrix = vectorizer.fit_transform(projection_text)
            if min(matrix.shape) >= 3:
                svd = TruncatedSVD(n_components=2,random_state=SEED)
                projection = pd.DataFrame(svd.fit_transform(matrix),index=projection_text.index,columns=["x","y"])
        except ValueError as error:
            if "empty vocabulary" not in str(error): raise
    if projection is not None:
        plt.figure(figsize=(10,6))
        if labels is not None:
            hue = labels.reindex(projection.index).astype("string").fillna("(tanpa label)")
            top = hue.value_counts().head(4).index
            hue = hue.where(hue.isin(top),"Lainnya (gabungan)")
            sns.scatterplot(data=projection,x="x",y="y",hue=hue,palette=[BLUE,YELLOW,INK,"#60758a","#b4c1cb"][:hue.nunique()],s=30,alpha=.65,linewidth=0)
        else: plt.scatter(projection.x,projection.y,color=BLUE,s=25,alpha=.6)
        finish(f"Peta leksikal · {len(projection):,} teks | variasi terjelaskan {svd.explained_variance_ratio_.sum():.1%}", "Komponen SVD 1", "Komponen SVD 2")
    else:
        print("Lewati: perlu >=3 dokumen dan fitur kosakata yang memadai.")

## 38 · Profil multilabel

Pakai list `TARGET` biner. Histogram jumlah label aktif per baris lengkap dan peta Jaccard antarlabel; lebih informatif daripada korelasi saja untuk label langka.

**Angka cepat · kardinalitas multilabel**

Hanya baris label lengkap. 0=negatif, NaN=unknown. Density=proporsi sel aktif pada matriks lengkap.

In [85]:
if len(targets)>1:
    y = df[targets].replace(LABEL_DROP,np.nan).apply(pd.to_numeric,errors="raise").dropna()
    if not y.stack().isin([0,1]).all(): raise ValueError("Label harus biner.")
    print(f"Baris lengkap={len(y):,}; cardinality={y.sum(axis=1).mean():.3f}; density={y.mean().mean():.3f}")
    show("Jumlah label aktif per baris",distribution(y.sum(axis=1)).sort_index())
    show("Kombinasi label",y.value_counts().head(TOP).rename("n").to_frame())
    intersection = y.T@y
    union = y.sum().to_numpy()[:,None]+y.sum().to_numpy()[None,:]-intersection.to_numpy()
    show("Jaccard label",intersection.div(np.where(union==0,np.nan,union)))

**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [86]:
if RUN_PLOTS:
    if len(targets)>1:
        y = df[targets].replace(LABEL_DROP,np.nan).apply(pd.to_numeric,errors="raise").dropna()
        if not y.stack().isin([0,1]).all(): raise ValueError("Label harus biner 0/1.")
        if len(y):
            bars(y.sum(axis=1).value_counts().sort_index(),"Jumlah label aktif per dokumen","Jumlah baris",YELLOW)
            intersection = y.T @ y
            total = y.sum().to_numpy()
            union = total[:,None]+total[None,:]-intersection.to_numpy()
            jaccard = intersection.div(np.maximum(union,1))
            heat(jaccard,"Jaccard antarlabel · irisan / gabungan")
    else:
        print("Lewati: perlu beberapa kolom target biner.")

## 39 · Transisi tag BIO

Pakai `TAGS`, list atau string dipisah spasi. Heatmap transisi berurutan dinormalisasi per tag asal. Nol berarti transisi tidak teramati, bukan otomatis tidak valid.

**Angka cepat · transisi BIO**

Hitungan dan % per tag asal. Persen dihitung sebelum membatasi ke TOP tag.

In [87]:
if has(TAGS):
    seqs = df[TAGS].dropna().map(lambda v:v.split() if isinstance(v,str) else list(v))
    counts = Counter((a,b) for seq in seqs for a,b in zip(seq,seq[1:]))
    if counts:
        table = pd.Series(counts).unstack(fill_value=0)
        rows,cols = table.sum(axis=1).nlargest(TOP).index,table.sum().nlargest(TOP).index
        show("Transisi jumlah",table.loc[rows,cols])
        show("Transisi % tag asal",table.div(table.sum(axis=1),axis=0).mul(100).loc[rows,cols])

**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [88]:
if RUN_PLOTS:
    if has(TAGS):
        seqs = df[TAGS].dropna().map(lambda v:v.split() if isinstance(v,str) else list(v))
        transitions = Counter((a,b) for seq in seqs for a,b in zip(seq,seq[1:]))
        if transitions:
            table = pd.Series(transitions).unstack(fill_value=0)
            keep = table.sum(axis=1).nlargest(15).index
            shown = table.div(table.sum(axis=1),axis=0).mul(100).reindex(index=keep)
            shown = shown.reindex(columns=table.sum().nlargest(15).index,fill_value=0)
            heat(shown,"Transisi BIO · baris asal → kolom berikut (%)",True)
    else:
        print("Lewati: isi TAGS.")

## 40 · Numerik per kategori

Pakai `VALUE`, `SOURCE`. Boxen menggambar kuantil bertingkat; titik kuning maksimal 300 sampel memperlihatkan variasi individual. Cocok untuk nilai engagement/harga/rating.

**Angka cepat · numerik per kategori**

Ganti VALUE/COL. Gagal konversi/NaN/infinite dikeluarkan dari statistik dan jumlahnya dilaporkan.

In [89]:
COL = SOURCE
if has(VALUE):
    values = pd.to_numeric(df[VALUE],errors="coerce").replace([np.inf,-np.inf],np.nan)
    print(f"Null/gagal/non-finite={values.isna().sum():,}; nol={values.eq(0).sum():,}; negatif={values.lt(0).sum():,}")
    show(f"Statistik {VALUE}",values.describe(percentiles=[.01,.5,.9,.95,.99]))
    if has(COL): show("Per kategori",values.groupby(df[COL],dropna=False).describe(percentiles=[.5,.95,.99]).sort_values("count",ascending=False).head(TOP))

Null/gagal/non-finite=93; nol=0; negatif=0

Statistik rating


,rating
count,1107.000
mean,2.938
std,1.421
min,1.000
1%,1.000
50%,3.000
90%,5.000
95%,5.000
99%,5.000
max,5.000



Per kategori


,count,mean,std,min,50%,95%,99%,max
source,,,,,,,,
aplikasi,581.0,2.988,1.440,1.0,3.0,5.0,5.0,5.0
sosmed,325.0,2.846,1.381,1.0,3.0,5.0,5.0,5.0
web,201.0,2.940,1.431,1.0,3.0,5.0,5.0,5.0


**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [90]:
if RUN_PLOTS:
    COL = SOURCE
    if has(VALUE) and has(COL):
        view = pd.DataFrame({"nilai":pd.to_numeric(df[VALUE],errors="coerce"),"kategori":df[COL]})
        view = view.replace([np.inf,-np.inf],np.nan).dropna()
        order = view.kategori.value_counts().head(8).index.tolist()
        if len(view) and order:
            plt.figure(figsize=(10,max(4,len(order)*.65)))
            sns.boxenplot(data=view,x="nilai",y="kategori",order=order,color=BLUE,k_depth="tukey")
            sns.stripplot(data=view.sample(min(300,len(view)),random_state=SEED),x="nilai",y="kategori",
                          order=order,color=YELLOW,size=3,jitter=.18,alpha=.7)
            finish(f"Distribusi {VALUE} per {COL}",VALUE,"")
    else:
        print("Lewati: perlu VALUE numerik dan kategori.")

## 41 · Dumbbell pergeseran panjang

Pakai `test`, `TEXT`. Bandingkan kuantil panjang kata train vs test pada satu sumbu. Pengukuran ini tidak menyimpulkan shift semantik.

**Angka cepat · pergeseran kuantil**

Seluruh baris setiap split; kosong dengan panjang 0. Selisih positif=kuantil test lebih panjang.

In [91]:
if has(TEXT) and test is not None and TEXT in test and len(test):
    q = [.01,.1,.25,.5,.75,.9,.95,.99]
    a = features.kata.quantile(q)
    b = test[TEXT].astype("string").fillna("").str.split().str.len().quantile(q)
    show("Kuantil panjang train-test",pd.DataFrame({"train":a,"test":b,"selisih_test_minus_train":b-a}))

**Grafik opsional** · berjalan hanya saat RUN_PLOTS=True.

In [92]:
if RUN_PLOTS:
    if has(TEXT) and test is not None and TEXT in test and len(test):
        quantiles = [.1,.25,.5,.75,.9,.95,.99]
        a = features["kata"].quantile(quantiles)
        b = test[TEXT].fillna("").astype(str).str.split().str.len().quantile(quantiles)
        fig,ax=plt.subplots(figsize=(10,5))
        ax.hlines(range(len(a)),np.minimum(a,b),np.maximum(a,b),color=INK,alpha=.25,lw=3)
        ax.scatter(a,range(len(a)),s=95,color=BLUE,label="Train",zorder=3)
        ax.scatter(b,range(len(b)),s=95,color=YELLOW,label="Test",zorder=3,edgecolor=INK,linewidth=.4)
        ax.set_yticks(range(len(a)),[f"P{int(q*100)}" for q in quantiles])
        ax.legend(frameon=False); ax.grid(axis="y",visible=False)
        finish("Pergeseran panjang per kuantil", "Jumlah kata", "Kuantil")
    else:
        print("Lewati: perlu teks train dan test.")

## Merakit notebook final

**Tidak perlu memakai semuanya.** Pilih chart untuk mendukung pertanyaan dan temuan.

- Minimum tabular: 04, 06, 14, 24, 40.
- Minimum NLP: 07, 09, 11, 15, 17, 18, 33.
- Pergeseran/split: 08, 19, 29, 30, 32, 41.
- Eksplorasi leksikal: 16, 28, 34, 35, 37.
- Task khusus: 20–23, 36, 38–39 sesuai kolom yang tersedia.

Semua template memakai setup 00–03. Bagian lainnya punya impor dan perhitungan lokal: tidak harus menjalankan chart sebelumnya. Sel 15 mendefinisikan helper khusus n-gram.

Untuk menyimpan visual pilihan, tambahkan `plt.savefig("chart.png", dpi=200, bbox_inches="tight")` sebelum `plt.show()`. Default hanya output inline di `.ipynb`. Sesuaikan judul dan satuan sebelum dipakai di laporan.


Untuk angka saja, pertahankan RUN_PLOTS=False; grafik dan SVD tidak dihitung. Semua sel angka memakai 00–03c, kemudian bisa dipilih tanpa menjalankan grafik. Untuk daftar lengkap, ganti .head(TOP) pada tabel yang dibutuhkan. Deteksi bahasa/kamus hanya diagnostik; hasil EDA tidak menghapus baris atau mengoreksi label otomatis.
